In [354]:
# 데이터 파이프라인
import pandas as pd
import pymysql
from sshtunnel import SSHTunnelForwarder
import os
from dotenv import load_dotenv
from datetime import datetime

# 알고리즘
from sklearn.cluster import KMeans

# 시각화
import folium
import random

# 데이터 처리
import ast
import calendar
import numpy as np
from math import radians

In [355]:
class AutoContainerGeneration:
    def __init__(self, version=1, debug=False):
        self.version = version
        self.debug = debug
        print(f"AutoContainerGeneration version: {version}")

        load_dotenv()

    # 공통 MySQL 데이터 추출 메서드
    def fetch_data(self, query, ssh_host, ssh_user, ssh_private_key, mysql_host, mysql_port, mysql_user, mysql_password, mysql_database):
        """
        MySQL 데이터를 추출하여 DF로 변환
        """
        try:
            with SSHTunnelForwarder(
                (ssh_host, 22),
                ssh_username=ssh_user,
                ssh_private_key=ssh_private_key,
                remote_bind_address=(mysql_host, mysql_port)
            ) as tunnel:
                print("SSH 터널 연결 성공")

                with pymysql.connect(
                    host='127.0.0.1',
                    user=mysql_user,
                    passwd=mysql_password,
                    db=mysql_database,
                    charset='utf8',
                    port=tunnel.local_bind_port,
                    cursorclass=pymysql.cursors.DictCursor) as conn:
                    with conn.cursor() as cur:
                        cur.execute(query)
                        results = cur.fetchall()
                        print("쿼리 실행 완료")

                        # 결과를 DF로 변환

                        df = pd.DataFrame(results)
                        return df

        except Exception as e:
            print(f"Error fetching data for: {e}")
            return None

    # 전체 데이터 추출
    def fetch_all_data(self):
        """
        MySQL 쿼리를 실행하여 Shipping Items와 Bunny Schedule 데이터를 DF로 변환
        """
        # SSH 및 MySQL 설정
        ssh_host = os.getenv("SSH_HOST_VER_1")
        ssh_user = os.getenv("SSH_USER")
        ssh_private_key = os.getenv(r"SSH_PRIVATE_KEY")
        

        mysql_host = os.getenv("MYSQL_HOST")
        mysql_port = 3306
        mysql_user = os.getenv("MYSQL_USER")
        mysql_password = os.getenv("MYSQL_PASSWORD")
        mysql_database = os.getenv("MYSQL_DATABASE")

        # 우편번호 그룹 폴리곤 데이터 쿼리
        # 평일
        regular_zipcode_groups_polygon_query = """
        SELECT region,
        group_name,
        ST_AsText(geometry) AS geometry,
        zipcodes
        from zipcode_groups_polygon
        where weekday = 0;
        """
        # 주말
        weekend_zipcode_groups_polygon_query = """
        SELECT region,
        group_name,
        ST_AsText(geometry) AS geometry,
        zipcodes
        from zipcode_groups_polygon
        where weekday = 1;
        """

        # 각 쿼리 결과를 DataFrame으로 가져오기

        df_regular_zipcode = self.fetch_data(regular_zipcode_groups_polygon_query, ssh_host, ssh_user, ssh_private_key,
                                                  mysql_host, mysql_port, mysql_user, mysql_password, mysql_database)
        df_weekend_zipcode = self.fetch_data(weekend_zipcode_groups_polygon_query, ssh_host, ssh_user, ssh_private_key,
                                                  mysql_host, mysql_port, mysql_user, mysql_password, mysql_database)

        return df_regular_zipcode, df_weekend_zipcode

In [356]:
if __name__ == "__main__":
    generator = AutoContainerGeneration(version=2, debug=True)
    df_regular, df_weekend = generator.fetch_all_data()

    # 각 DataFrame 확인
    print("DF 생성 완료")

AutoContainerGeneration version: 2
SSH 터널 연결 성공
쿼리 실행 완료
SSH 터널 연결 성공
쿼리 실행 완료
DF 생성 완료


In [357]:
workflow_day_shipping_items_df = pd.read_csv('../git_csv/20250329_오버랩제외_전체물량.csv')

workflow_day_bunny_df = pd.read_csv('../git_csv/20250329_고정버니.csv')

In [358]:
def group_and_map_zipcodes(zipcode_groups, shipping_csv_path):
    
    result_gdf = zipcode_groups

    # 우편번호-그룹 매핑 생성
    zipcode_to_group = {}
    for _, row in result_gdf.iterrows():
        group = row['group_name']
        # 문자열 형태의 zipcodes를 리스트로 변환
        zipcodes_group = ast.literal_eval(row['zipcodes'])
        for zipcode in zipcodes_group:
            zipcode_to_group[zipcode] = group

    # 배송 데이터 로드 및 그룹 매핑
    df_shipping = shipping_csv_path
    df_shipping['group'] = df_shipping['zipcode'].map(zipcode_to_group)
    df_shipping = df_shipping[~df_shipping['group'].isna()]
    
    return result_gdf, df_shipping

In [359]:
def visualize_clusters(updated_df):
    # 지도 초기화
    map_center = [updated_df['lat'].mean(), updated_df['lng'].mean()]  # 물품의 평균 좌표를 중심으로 설정
    delivery_map = folium.Map(location=map_center, zoom_start=12)
    
    # 'group' 컬럼의 고유한 값들을 가져오되, NaN은 제외
    unique_groups = updated_df['cluster_label'].dropna().unique()
    color_map = { group: f"#{random.randint(0, 0xFFFFFF):06x}" for group in unique_groups }
    
    # 기본 색상 설정 (NaN인 경우 사용할 색상)
    default_color = "#808080"

    # 동그라미 마커 추가
    for _, row in updated_df.iterrows():
        Area = row['Area']
        latitude = row['lat']
        longitude = row['lng']
        item_uuid = row['shipping_uuid']
        zipcode = row['zipcode']
        Type = row['driver_type']
        cluster_label = row['cluster_label']
        code = row['code']
        

        # group이 NaN인지 확인하여 기본 색상 지정
        if Type == 'BLUE':
            color = default_color
        else:
            color = color_map.get(cluster_label, default_color)
        

        tooltip = f"Area: {Area}<br>Shipping_uuid: {item_uuid}<br>lat: {latitude}<br>lng: {longitude}<br>Zipcode: {zipcode}<br>Type: {Type}<br>cluster_label: {cluster_label}<br>Sector_code: {code}"
        folium.CircleMarker(
            location=[latitude, longitude],
            radius=10,  # 동그라미 크기
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.7,
            tooltip=tooltip
        ).add_to(delivery_map)

    return delivery_map

In [360]:
# 헬퍼 함수

# 1) 그룹별 중심 좌표 재계산
def recalc_group_centroids(df, group_col='group', lat_col='lat', lng_col='lng'):
    """
    DataFrame 내 group별 위경도(lat, lng) 평균을 구하여 centroid(중심좌표) 정보를 dict 형태로 반환
    """
    centroids = df.groupby(group_col).apply(
        lambda sub_df: (sub_df[lat_col].mean(), sub_df[lng_col].mean())
    ).to_dict()
    return centroids

# 2) 날짜 반환
def is_weekend(year, month, day):
    return calendar.weekday(year, month, day) >= 5

# 3) 버니스케줄과 물품데이터 매핑
def normalize_region(region):
    """
    workflow_day_bunny_df의 Area는 대부분 뒤에 '구'나 '시'가 없는 형식임.
    단, '일산서구', '일산동구'는 예외로 그대로 사용.
    """
    if isinstance(region, list):
        return [normalize_region(r) for r in region]
    
    if region in ["일산서구", "일산동구", "중구"]:
        return region
    if region.endswith("구") or region.endswith("시"):
        return region[:-1]
    return region

# 4) Kmeans 함수
def split_group_kmeans_2clusters(df, group):
    """
    df에서 특정 group에 속한 주문들을 K-Means(n_clusters=2)로 분할하고,
    클러스터 라벨과 각 클러스터의 주문 건수를 반환합니다.
    반환값: (grp_orders, cluster_counts, label_a, count_a, label_b, count_b)
    """
    grp_orders = df[df['group'] == group].copy()
    grp_orders['lat_rad'] = grp_orders['lat'].apply(radians)
    grp_orders['lng_rad'] = grp_orders['lng'].apply(radians)
    coords = grp_orders[['lat_rad', 'lng_rad']].to_numpy()
    if len(coords) < 2:
        return grp_orders, {}, None, 0, None, 0
    km = KMeans(n_clusters=2, init='k-means++', random_state=42)
    cluster_labels = km.fit_predict(coords)
    grp_orders['cluster'] = cluster_labels
    cluster_counts = grp_orders['cluster'].value_counts()
    if len(cluster_counts) < 2:
        label_a = cluster_counts.index[0]
        return grp_orders, cluster_counts, label_a, cluster_counts.iloc[0], None, 0
    label_a = cluster_counts.index[0]
    count_a = cluster_counts.iloc[0]
    label_b = cluster_counts.index[1]
    count_b = cluster_counts.iloc[1]
    return grp_orders, cluster_counts, label_a, count_a, label_b, count_b

# 5) 주문 건 이동
def move_orders(df, indices, target_group, clear_driver=True):
    """
    df에서 indices에 해당하는 주문들을 target_group으로 이동합니다.
    clear_driver=True이면, 이동 시 'driver_type'을 np.nan으로 초기화합니다.
    """
    if clear_driver:
        df.loc[indices, 'driver_type'] = np.nan
        df.loc[indices, 'driver_code'] = np.nan
    df.loc[indices, 'group'] = target_group
    print(f"[move_orders] Moved {len(indices)} orders to group [{target_group}].")

# 6) 버니 배정
def assign_driver_to_group(df, group_name, driver_df, assigned_idx, leftover_count, label=""):
    """
    driver_df의 assigned_idx번째 기사를 df에서 group_name에 배정.
    배정 후 assigned_idx / leftover_count 갱신.
    
    반환: (df, assigned_idx, leftover_count)
    """
    if assigned_idx >= len(driver_df):
        print(f"[assign_driver_to_group] 더 이상 {label} 드라이버가 없습니다.")
        return df, assigned_idx, leftover_count
    
    driver = driver_df.iloc[assigned_idx]
    df.loc[df['group'] == group_name, 'driver_type'] = driver['Type']
    
    assigned_idx += 1
    leftover_count -= 1
    
    return df, assigned_idx, leftover_count

# 7) 범위
def in_range(x, low, high):
    return (x >= low) and (x < high)

# 8) 기존 중심점에서 데이터 이동 말고 가까운 데이터들 가져오기
def min_dist_to_group(row, group_points):
    # 유클리드 예시 (필요시 Haversine 대체)
    dists = ((group_points['lat'] - row['lat'])**2 
           + (group_points['lng'] - row['lng'])**2)
    return np.sqrt(dists.min())

In [361]:
def isolate_white_clusters(df, group_name):
    iteration = 1
    group_stack = [group_name]  # 처리할 그룹을 스택에 저장

    while group_stack:
        current_group = group_stack.pop()
        group_orders = df[df['group'] == current_group].copy()
        total_count = group_orders['shipping_uuid'].count()

        if total_count >= 60:
            print(f"{current_group}의 주문 수가 {total_count}건")

        # (1) 40건 미만이면 화이트 처리 중단
        if total_count < 40:
            print(f"그룹 {current_group}의 주문 수가 {total_count}건으로 40 미만이어서 추가 분리 종료")
            continue

        if 60 <= total_count <= 69:
            print(f"{total_count}건 이므로 한 클러스터 30으로 조정")

            # --- K-Means 2클러스터링 ---
            grp_orders, cluster_counts, label_a, count_a, label_b, count_b = split_group_kmeans_2clusters(df, current_group)

            # 실제 group_orders에 cluster 정보를 반영 (반복 조정 위해)
            if len(cluster_counts) < 2 or label_b is None:
                print("클러스터가 2개 미만이거나 하나뿐이어서 처리 불가, 계속 진행")
                continue

            print(f"클러스터링 결과: {cluster_counts.to_dict()}")
            group_orders['cluster'] = grp_orders['cluster']

            # 작은/큰 클러스터 식별
            if count_a < count_b:
                label_small, label_large = label_a, label_b
            elif count_a > count_b:
                label_small, label_large = label_b, label_a
            else:
                print("두 클러스터 사이즈 동일 -> 강제로 label_small=0, label_large=1")
                sorted_idx = sorted(cluster_counts.index)
                label_small, label_large = sorted_idx[0], sorted_idx[1]

            # 반복적으로 작은 클러스터를 30건으로 맞추기
            while True:
                cluster_counts = group_orders['cluster'].value_counts()
                if len(cluster_counts) < 2:
                    break

                label_a2 = cluster_counts.index[0]
                count_a2 = cluster_counts.iloc[0]
                label_b2 = cluster_counts.index[1]
                count_b2 = cluster_counts.iloc[1]

                # 다시 작은/큰 라벨 식별
                if count_a2 < count_b2:
                    label_small, label_large = label_a2, label_b2
                elif count_a2 > count_b2:
                    label_small, label_large = label_b2, label_a2
                else:
                    print("작은/큰 클러스터 사이즈 동일 -> tie-break")
                    sorted_idx = sorted(cluster_counts.index)
                    label_small, label_large = sorted_idx[0], sorted_idx[1]

                count_small = cluster_counts[label_small]
                count_large = cluster_counts[label_large]

                # 두 클러스터가 모두 30~39면 각각 WHITE 그룹으로 마무리
                if (30 <= count_a2 < 40) and (30 <= count_b2 < 40):
                    new_white_group_a = f"{current_group}_{label_a2}_WHITE"
                    new_white_group_b = f"{current_group}_{label_b2}_WHITE"

                    idx_a = group_orders[group_orders['cluster'] == label_a2].index
                    idx_b = group_orders[group_orders['cluster'] == label_b2].index

                    move_orders(df, idx_a, new_white_group_a, clear_driver=False)
                    move_orders(df, idx_b, new_white_group_b, clear_driver=False)

                    print(f"[화이트 분리 완료] 그룹 {new_white_group_a} {len(idx_a)}건, {new_white_group_b} {len(idx_b)}건 생성, driver_type='WHITE'로 업데이트")
                    break

                # 정확히 30이면 루프 종료
                if count_small == 30:
                    break

                elif count_small < 30:
                    # 작은 클러스터 부족 -> 큰 클러스터에서 가져오기
                    deficit = 30 - count_small
                    available = count_large - 30
                    if available <= 0:
                        print(f"조정 불가: 큰 클러스터({label_large})에 여분 주문이 없어 이동 불가")
                        break
                    move_count = min(deficit, available)

                    print(f"작은 클러스터 부족: {deficit}건, donor 클러스터에서 {move_count}건 이동 시도")

                    donor_orders = group_orders[group_orders['cluster'] == label_large].copy()

                    # 작은 클러스터에 속한 좌표들
                    small_cluster_df = group_orders[group_orders['cluster'] == label_small][["lat","lng"]]

                    # min_dist_to_group 사용
                    donor_orders['dist_to_small'] = donor_orders.apply(
                        lambda row: min_dist_to_group(row, small_cluster_df),
                        axis=1
                    )
                    donor_orders.sort_values('dist_to_small', ascending=True, inplace=True)
                    orders_to_move = donor_orders.head(move_count)

                    if len(orders_to_move) == 0:
                        print("이동할 주문이 없어 추가 조정 불가능")
                        break

                    group_orders.loc[orders_to_move.index, 'cluster'] = label_small

                else:
                    # 작은 클러스터 초과 -> 큰 클러스터로 이동
                    surplus = count_small - 30
                    print(f"작은 클러스터 초과: {surplus}건, 큰 클러스터로 이동 시도")

                    small_orders = group_orders[group_orders['cluster'] == label_small].copy()
                    large_centroid = group_orders[group_orders['cluster'] == label_large][['lat', 'lng']].mean().values

                    # 가장 가까운 순으로 surplus건 이동
                    small_orders['dist_to_large'] = np.sqrt(
                        (small_orders['lat'] - large_centroid[0])**2
                        + (small_orders['lng'] - large_centroid[1])**2
                    )
                    small_orders.sort_values('dist_to_large', ascending=True, inplace=True)
                    orders_to_move = small_orders.head(surplus)

                    if len(orders_to_move) == 0:
                        print("이동할 주문이 없어 추가 조정 불가능")
                        break

                    group_orders.loc[orders_to_move.index, 'cluster'] = label_large

                # 이동 후 상황 확인
                cluster_counts = group_orders['cluster'].value_counts()
                print(f"이동 후 클러스터 분포: {cluster_counts.to_dict()}")

                if label_small not in cluster_counts:
                    print("label_small 클러스터가 사라져서 종료")
                    break

                count_small = cluster_counts.get(label_small, 0)
                if count_small == 30:
                    break

            # 작은 클러스터가 30건이면 화이트 확정
            count_small = group_orders['cluster'].value_counts().get(label_small, 0)
            if count_small == 30:
                new_white_group = f"{current_group}_WHITE"
                idx_white = group_orders[group_orders['cluster'] == label_small].index

                # 그룹 이동
                move_orders(df, idx_white, new_white_group, clear_driver=False)

                print(f"[화이트 분리 완료] 그룹 {new_white_group} (30건) 생성, driver_type='WHITE'로 업데이트")
                iteration += 1

                # 남은(큰) 클러스터 처리
                idx_remaining = group_orders[group_orders['cluster'] == label_large].index
                remaining_count = df.loc[idx_remaining, 'shipping_uuid'].count()
                if remaining_count < 40:
                    print(f"남은 그룹의 주문 수가 {remaining_count}건으로 40 미만이어서 추가 분리 종료")
                    new_remaining_group = f"{current_group}_R"
                    move_orders(df, idx_remaining, new_remaining_group, clear_driver=False)
                    print(f"남은 주문 {remaining_count}건은 그룹 {new_remaining_group}로 업데이트")
                    continue

        if 70 <= total_count <= 78:
            print(f"{total_count}건 이므로 큰 클러스터 39로 조정")

            grp_orders, cluster_counts, label_a, count_a, label_b, count_b = split_group_kmeans_2clusters(df, current_group)
            if len(cluster_counts) < 2 or label_b is None:
                print("클러스터가 2개 미만이라 처리 불가")
                continue

            print(f"클러스터링 결과: {cluster_counts.to_dict()}")
            group_orders['cluster'] = grp_orders['cluster']

            # 작은/큰 클러스터 구분
            if count_a < count_b:
                label_small, label_large = label_a, label_b
            elif count_a > count_b:
                label_small, label_large = label_b, label_a
            else:
                print("두 클러스터 사이즈 동일 -> 강제로 label_small=0, label_large=1")
                sorted_idx = sorted(cluster_counts.index)
                label_small, label_large = sorted_idx[0], sorted_idx[1]

            # 큰 클러스터를 39건으로 조정
            while True:
                cluster_counts2 = group_orders['cluster'].value_counts()
                if len(cluster_counts2) < 2:
                    break

                la = cluster_counts2.index[0]
                ca = cluster_counts2.iloc[0]
                lb = cluster_counts2.index[1]
                cb = cluster_counts2.iloc[1]

                if ca < cb:
                    label_small, label_large = la, lb
                elif ca > cb:
                    label_small, label_large = lb, la
                else:
                    print("작은/큰 클러스터 사이즈 동일 -> tie-break")
                    sorted_idx = sorted(cluster_counts2.index)
                    label_small, label_large = sorted_idx[0], sorted_idx[1]

                count_small = cluster_counts2[label_small]
                count_large = cluster_counts2[label_large]

                if count_large == 39:
                    break
                elif count_large < 39:
                    deficit = 39 - count_large
                    available = count_small - 30
                    if available <= 0:
                        print(f"조정 불가: 작은 클러스터({label_small})에 여분 주문이 없어 이동 불가")
                        break
                    move_count = min(deficit, available)
                    print(f"작은 클러스터 부족: {deficit}건 -> {move_count}건 이동")

                    donor_orders = group_orders[group_orders['cluster'] == label_small].copy()
                    large_cluster_df = group_orders[group_orders['cluster'] == label_large][["lat","lng"]]

                    donor_orders['dist_to_large'] = donor_orders.apply(
                        lambda row: min_dist_to_group(row, large_cluster_df),
                        axis=1
                    )
                    donor_orders.sort_values('dist_to_large', ascending=True, inplace=True)
                    orders_to_move = donor_orders.head(move_count)

                    if len(orders_to_move) == 0:
                        print("이동할 주문이 없어 추가 조정 불가능")
                        break

                    group_orders.loc[orders_to_move.index, 'cluster'] = label_large

                else:  # count_large > 39
                    surplus = count_large - 39
                    print(f"큰 클러스터 초과: {surplus}건 -> 작은 클러스터 이동")

                    large_orders = group_orders[group_orders['cluster'] == label_large].copy()
                    small_centroid = group_orders[group_orders['cluster'] == label_small][['lat', 'lng']].mean().values

                    large_orders['dist_to_small'] = np.sqrt(
                        (large_orders['lat'] - small_centroid[0])**2
                        + (large_orders['lng'] - small_centroid[1])**2
                    )
                    large_orders.sort_values('dist_to_small', ascending=True, inplace=True)
                    orders_to_move = large_orders.head(surplus)

                    if len(orders_to_move) == 0:
                        print("이동할 주문이 없어 추가 조정 불가능")
                        break

                    group_orders.loc[orders_to_move.index, 'cluster'] = label_small

                print(f"이동 후 클러스터 분포: {group_orders['cluster'].value_counts().to_dict()}")

                if label_small not in group_orders['cluster'].value_counts():
                    print("label_small 클러스터가 사라져서 종료")
                    break

                if group_orders['cluster'].value_counts().get(label_large, 0) == 39:
                    break

            # 큰 클러스터가 39건이면 화이트 확정
            count_large = group_orders['cluster'].value_counts().get(label_large, 0)
            if count_large == 39:
                new_white_group = f"{current_group}_WHITE_{iteration}"
                idx_white = group_orders[group_orders['cluster'] == label_large].index

                move_orders(df, idx_white, new_white_group, clear_driver=False)
                print(f"[화이트 분리 완료] 그룹 {new_white_group} (39건) 생성, driver_type='WHITE'로 업데이트")

                iteration += 1

                # 작은 클러스터 처리
                idx_small = group_orders[group_orders['cluster'] == label_small].index
                remaining_count = df.loc[idx_small, 'shipping_uuid'].count()
                if remaining_count < 40:
                    print(f"남은 그룹의 주문 수가 {remaining_count}건으로 40 미만이어서 추가 분리 종료")
                    new_remaining_group = f"{current_group}_R_WHITE"
                    move_orders(df, idx_small, new_remaining_group, clear_driver=False)
                    print(f"남은 주문 {remaining_count}건은 그룹 {new_remaining_group}로 업데이트")
                    continue

        print(f"\n[화이트 처리] 그룹 {current_group} 총 주문 수: {total_count}건. 클러스터링 시도...")

        grp_orders, cluster_counts, label_a, count_a, label_b, count_b = split_group_kmeans_2clusters(df, current_group)
        if len(cluster_counts) < 2 or label_b is None:
            print("클러스터가 2개 미만이라 스킵")
            continue

        print(f"클러스터링 결과: {cluster_counts.to_dict()}")
        group_orders['cluster'] = grp_orders['cluster']

        # 클러스터가 둘 다 40 이상이면 각각 스택에 다시 추가
        if count_a >= 40 and count_b >= 40:
            new_group_a = f"{current_group}_SPLIT_{label_a}"
            new_group_b = f"{current_group}_SPLIT_{label_b}"

            idx_a = group_orders[group_orders['cluster'] == label_a].index
            idx_b = group_orders[group_orders['cluster'] == label_b].index

            move_orders(df, idx_a, new_group_a, clear_driver=False)
            move_orders(df, idx_b, new_group_b, clear_driver=False)

            print(f"→ 두 클러스터가 모두 40건 이상. 새로운 그룹 {new_group_a}, {new_group_b} 스택에 추가")
            group_stack.append(new_group_a)
            group_stack.append(new_group_b)
            continue

        # 둘 다 20~39면 둘 다 WHITE 배정
        if 20 <= count_a < 40 and 20 <= count_b < 40:
            new_white_group_a = f"{current_group}_WHITE_{label_a}"
            new_white_group_b = f"{current_group}_WHITE_{label_b}"

            idx_a = group_orders[group_orders['cluster'] == label_a].index
            idx_b = group_orders[group_orders['cluster'] == label_b].index

            move_orders(df, idx_a, new_white_group_a, clear_driver=False)
            move_orders(df, idx_b, new_white_group_b, clear_driver=False)

            print(f"클러스터 초기 그룹 물량 20건이상 40건 미만이므로 그룹 {new_white_group_a}, {new_white_group_b} driver_type='WHITE'로 업데이트")
            continue

        # 한쪽이 20~39, 다른 쪽이 40 이상
        if 20 <= count_a < 40 and count_b >= 40:
            new_white_group_a = f"{current_group}_WHITE_{label_a}"

            idx_a = group_orders[group_orders['cluster'] == label_a].index
            idx_b = group_orders[group_orders['cluster'] == label_b].index

            move_orders(df, idx_a, new_white_group_a, clear_driver=False)
            move_orders(df, idx_b, f"{current_group}_R", clear_driver=False)

            print("A 화이트 배정")
            print(f"클러스터 초기 그룹 물량 20건이상 40건 미만이므로 그룹 {new_white_group_a} driver_type='WHITE'로 업데이트")

            current_group = f"{current_group}_R"
            group_stack.append(current_group)
            continue

        if 20 <= count_b < 40 and count_a >= 40:
            new_white_group_b = f"{current_group}_WHITE_{label_b}"

            idx_a = group_orders[group_orders['cluster'] == label_a].index
            idx_b = group_orders[group_orders['cluster'] == label_b].index

            move_orders(df, idx_b, new_white_group_b, clear_driver=False)
            move_orders(df, idx_a, f"{current_group}_R", clear_driver=False)

            print("B 화이트 배정")
            print(f"클러스터 초기 그룹 물량 20건이상 40건 미만이므로 그룹 {new_white_group_b} driver_type='WHITE'로 업데이트")

            current_group = f"{current_group}_R"
            group_stack.append(current_group)
            continue

        if count_a < count_b:
            label_small, label_large = label_a, label_b
        elif count_a > count_b:
            label_small, label_large = label_b, label_a
        else:
            print("두 클러스터 사이즈 동일 -> 강제로 label_small=0, label_large=1")
            sorted_idx = sorted(cluster_counts.index)
            label_small, label_large = sorted_idx[0], sorted_idx[1]

        while True:
            cluster_counts2 = group_orders['cluster'].value_counts()
            if len(cluster_counts2) < 2:
                break

            la = cluster_counts2.index[0]
            ca = cluster_counts2.iloc[0]
            lb = cluster_counts2.index[1]
            cb = cluster_counts2.iloc[1]

            if ca < cb:
                label_small, label_large = la, lb
            elif ca > cb:
                label_small, label_large = lb, la
            else:
                print("작은/큰 클러스터 사이즈 동일 -> tie-break")
                sorted_idx = sorted(cluster_counts2.index)
                label_small, label_large = sorted_idx[0], sorted_idx[1]

            count_small = cluster_counts2[label_small]
            count_large = cluster_counts2[label_large]

            if count_small == 20:
                break
            elif count_small < 20:
                deficit = 20 - count_small
                available = count_large - 20
                if available <= 0:
                    print(f"조정 불가: 큰 클러스터({label_large})에 여분 주문이 없어 이동 불가")
                    break

                move_count = min(deficit, available)
                print(f"작은 클러스터 부족: {deficit}건, donor 클러스터에서 {move_count}건 이동 시도")

                donor_orders = group_orders[group_orders['cluster'] == label_large].copy()
                small_cluster_df = group_orders[group_orders['cluster'] == label_small][["lat","lng"]]

                donor_orders['dist_to_small'] = donor_orders.apply(
                    lambda row: min_dist_to_group(row, small_cluster_df),
                    axis=1
                )
                donor_orders.sort_values('dist_to_small', ascending=True, inplace=True)
                orders_to_move = donor_orders.head(move_count)

                if len(orders_to_move) == 0:
                    print("이동할 주문이 없어 추가 조정 불가능")
                    break

                group_orders.loc[orders_to_move.index, 'cluster'] = label_small

            else:
                # 작은 클러스터 초과 -> 큰 클러스터로 이동
                surplus = count_small - 20
                print(f"작은 클러스터 초과: {surplus}건, 큰 클러스터로 이동 시도")

                donor_orders = group_orders[group_orders['cluster'] == label_small].copy()
                large_cluster_df = group_orders[group_orders['cluster'] == label_large][["lat","lng"]]

                donor_orders['dist_to_large'] = donor_orders.apply(
                    lambda row: min_dist_to_group(row, large_cluster_df),
                    axis=1
                )
                donor_orders.sort_values('dist_to_large', ascending=True, inplace=True)
                orders_to_move = donor_orders.head(surplus)

                if len(orders_to_move) == 0:
                    print("이동할 주문이 없어 추가 조정 불가능")
                    break

                group_orders.loc[orders_to_move.index, 'cluster'] = label_large

            cluster_counts2 = group_orders['cluster'].value_counts()
            print(f"이동 후 클러스터 분포: {cluster_counts2.to_dict()}")

            if label_small not in cluster_counts2:
                print("label_small 클러스터가 사라져서 종료")
                break

            if cluster_counts2.get(label_small, 0) == 20:
                break

        # 작은 클러스터가 20건이면 화이트로 확정
        final_count_small = group_orders['cluster'].value_counts().get(label_small, 0)
        if final_count_small == 20:
            new_white_group = f"{current_group}_WHITE_{iteration}"
            idx_white = group_orders[group_orders['cluster'] == label_small].index

            move_orders(df, idx_white, new_white_group, clear_driver=False)
            print(f"[화이트 분리 완료] 그룹 {new_white_group} (20건) 생성, driver_type='WHITE'로 업데이트")

            iteration += 1

            # 나머지(큰 클러스터) 처리
            idx_remaining = group_orders[group_orders['cluster'] == label_large].index
            remaining_count = df.loc[idx_remaining, 'shipping_uuid'].count()
            if remaining_count < 40:
                print(f"남은 그룹의 주문 수가 {remaining_count}건으로 40 미만이어서 추가 분리 종료")
                continue

            new_remaining_group = f"{current_group}_R"
            move_orders(df, idx_remaining, new_remaining_group, clear_driver=False)
            print(f"남은 주문 {remaining_count}건은 그룹 {new_remaining_group}로 업데이트")

            # 스택에 새 그룹 push → 다음 while loop에서 처리
            group_stack.append(new_remaining_group)

        else:
            print(f"화이트 분리 불가: 작은 클러스터가 20건이 아닙니다. (현재 {final_count_small}건)")
            continue

    return df

In [362]:
def assign_fixed_drivers(df_shipping_region, fix_region_workflow_day_bunny_df):
    for col in ['driver_type', 'driver_code']:
        if col not in df_shipping_region.columns:
            df_shipping_region[col] = np.nan

    group_centroids = recalc_group_centroids(df_shipping_region)
    # 1) 버니 우선순위 매핑 및 정렬
    type_priority = {'YELLOW': 1, 'RAINBOW': 2, 'ORANGE': 3}
    fix_region_workflow_day_bunny_df['type_prio'] = fix_region_workflow_day_bunny_df['Type'].map(type_priority)
    fix_region_workflow_day_bunny_df.sort_values(['type_prio'], inplace=True)

    # 2) Y/R와 ORANGE 드라이버 분리
    non_orange_drivers = fix_region_workflow_day_bunny_df[fix_region_workflow_day_bunny_df['Type'] != 'ORANGE']
    orange_drivers = fix_region_workflow_day_bunny_df[fix_region_workflow_day_bunny_df['Type'] == 'ORANGE']
    assigned_non_orange = 0
    assigned_orange = 0
    leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
    leftover_orange = len(orange_drivers) - assigned_orange
    # 1차 배정
    # (A) YELLOW/RAINBOW 배정: 그룹 주문 수 40~50건인 그룹
    group_counts = df_shipping_region.groupby('group')['shipping_uuid'].count()
    yellow_rainbow_groups = group_counts[(group_counts >= 40) & (group_counts < 50)].index

    print(f"[1차 배정] Y/R driver count: {len(non_orange_drivers)}")
    print(f"[1차 배정] ORANGE driver count: {len(orange_drivers)}")

    for grp in yellow_rainbow_groups:
        if assigned_non_orange >= len(non_orange_drivers):
            break
        
        # 새 그룹명 예: 기존 grp + "_Y/R"
        new_grp = f"{grp}_Y/R"
        
        # 주문들을 새 그룹으로 이동 (여기서는 단순히 group만 바꿈)
        indices = df_shipping_region[df_shipping_region['group'] == grp].index
        df_shipping_region.loc[indices, 'group'] = new_grp
        
        # 드라이버 배정 (driver_type, driver_code 설정)
        df_shipping_region, assigned_non_orange, leftover_non_orange = assign_driver_to_group(
                            df_shipping_region,
                            new_grp,
                            non_orange_drivers,
                            assigned_non_orange,
                            leftover_non_orange,
                            label="Y/R"
        )
        
        print(f"[1차 배정] Group {grp} reassigned as {new_grp} and driver assigned.")
        
        # 그룹 중심점 재계산 (필요 시)
        group_centroids = recalc_group_centroids(df_shipping_region)

    # 남은 Y/R 드라이버 수
    # leftover_non_orange = non_orange_drivers - assigned_non_orange
    print(f"[1차 배정] Leftover Y/R drivers: {leftover_non_orange}")


    # (B) ORANGE 배정: 그룹 주문 수 20~29건인 그룹
    group_counts = df_shipping_region.groupby('group')['shipping_uuid'].count()
    orange_groups = group_counts[(group_counts >= 20) & (group_counts <= 29)].index

    for grp in orange_groups:
        if assigned_orange >= len(orange_drivers):
            break
        
        new_grp = f"{grp}_O"
        indices = df_shipping_region[df_shipping_region['group'] == grp].index
        df_shipping_region.loc[indices, 'group'] = new_grp
        
        df_shipping_region, assigned_orange, leftover_orange = assign_driver_to_group(
                            df_shipping_region,
                            new_grp,
                            orange_drivers,
                            assigned_orange,
                            leftover_orange,
                            label="ORANGE"
        )

        print(f"[1차 배정] Group {grp} reassigned as {new_grp} and driver assigned.")
        
        # 그룹 중심점 재계산 (필요 시)
        group_centroids = recalc_group_centroids(df_shipping_region)

    # 남은 ORANGE 드라이버 수
    # leftover_orange = len(orange_drivers) - assigned_orange
    print(f"[1차 배정] Leftover ORANGE drivers: {leftover_orange}")


    # ---------------------------------------------------------------------------
    # 2차 배정: Y/R 2차 재배정 (주문 수 60건 이상인 그룹 대상)
    yr_group_counter = {}
    stop_all = False
    print("### [Y/R 2차 재배정] ###")

    for _ in range(len(non_orange_drivers)):
        if leftover_non_orange <= 0:
            stop_all = True
            break

        unassigned_df = df_shipping_region[df_shipping_region['driver_type'].isna()]
        group_counts = unassigned_df.groupby('group')['shipping_uuid'].count()

        # 60건 이상인 그룹을 대상으로 처리
        extra_unassigned_max_groups = group_counts[group_counts >= 60].index.tolist()
        print(f"배정되지 않은 그룹 {extra_unassigned_max_groups}")
        print(f"[Y/R 2차 재배정] 남은 Y/R 버니: {leftover_non_orange}명")
        
        if not extra_unassigned_max_groups:
            break

        for grp in extra_unassigned_max_groups:
            if leftover_non_orange <= 0:
                stop_all = True
                break

            # 스택(LIFO)으로 관리
            group_stack = [grp]

            while group_stack and not stop_all:
                current_group = group_stack.pop()
                
                # 해당 group의 실제 주문 수 확인
                current_count = df_shipping_region[df_shipping_region['group'] == current_group].shape[0]
                if current_count < 60:
                    # 60건 미만이면 스킵
                    continue

                print(f"[Y/R 2차 재배정] 그룹 [{current_group}] (주문수: {current_count})에서 클러스터 추출 시도")

                grp_orders, cluster_counts_res, label_a, count_a, label_b, count_b = \
                    split_group_kmeans_2clusters(df_shipping_region, current_group)

                # 클러스터가 2개 미만이면 스킵
                if len(cluster_counts_res) < 2 or label_b is None:
                    continue

                # 더 편하게 쓰기 위해 local 변수에 재저장
                cluster_counts = cluster_counts_res
                # 큰/작은 클러스터 식별
                if cluster_counts.iloc[0] <= cluster_counts.iloc[1]:
                    smaller_cluster = cluster_counts.index[0]  # ex) label_a
                    larger_cluster = cluster_counts.index[1]  # ex) label_b
                else:
                    smaller_cluster = cluster_counts.index[1]
                    larger_cluster = cluster_counts.index[0]

                print(f"  → 그룹 [{current_group}] 클러스터 결과: "
                    f"클러스터 {label_a} ({cluster_counts[label_a]}건), "
                    f"클러스터 {label_b} ({cluster_counts[label_b]}건)")

                # 클러스터별 인덱스
                indices_a = grp_orders[grp_orders['cluster'] == label_a].index
                indices_b = grp_orders[grp_orders['cluster'] == label_b].index

                if 55 < count_a < 60 and count_b >= 60:
                    new_group_a = f"{current_group}_R_{label_a}"
                    new_group_b = f"{current_group}_SPLIT_{label_b}"

                    # 그룹 이동 (driver_type은 초기화하지 않고 그대로 유지 → clear_driver=False)
                    move_orders(df_shipping_region, indices_a, new_group_a, clear_driver=False)
                    move_orders(df_shipping_region, indices_b, new_group_b, clear_driver=False)

                    print(f"→ 클러스터{label_a}: remain, 클러스터{label_b}: stack에 재추가 (다시 클러스터링)")

                    group_stack.append(new_group_b)
                    continue

                if 55 < count_b < 60 and count_a >= 60:
                    new_group_b = f"{current_group}_R_{label_b}"
                    new_group_a = f"{current_group}_SPLIT_{label_a}"

                    move_orders(df_shipping_region, indices_b, new_group_b, clear_driver=False)
                    move_orders(df_shipping_region, indices_a, new_group_a, clear_driver=False)

                    print(f"→ 클러스터{label_b}: remain, 클러스터{label_a}: stack에 재추가 (다시 클러스터링)")

                    group_stack.append(new_group_a)
                    continue

                if count_a >= 60 and count_b >= 60:
                    new_group_a = f"{current_group}_SPLIT_{label_a}"
                    new_group_b = f"{current_group}_SPLIT_{label_b}"

                    move_orders(df_shipping_region, indices_a, new_group_a, clear_driver=False)
                    move_orders(df_shipping_region, indices_b, new_group_b, clear_driver=False)

                    print(f"→ 두 클러스터가 모두 60건 이상. 새로운 그룹 {new_group_a}, {new_group_b} → stack에 추가")

                    group_stack.append(new_group_a)
                    group_stack.append(new_group_b)
                    continue

                if 40 <= count_a <= 55 and count_b >= 60:
                    new_YR_group_a = f"{current_group}_Y/R_{label_a}"
                    new_big_grp = f"{current_group}_remain"

                    # 그룹 이동
                    move_orders(df_shipping_region, indices_a, new_YR_group_a, clear_driver=False)
                    move_orders(df_shipping_region, indices_b, new_big_grp, clear_driver=False)

                    if assigned_non_orange < len(non_orange_drivers):
                        driver_before = non_orange_drivers.iloc[assigned_non_orange]
                        df_shipping_region, assigned_non_orange, leftover_non_orange = assign_driver_to_group(
                            df_shipping_region,
                            new_YR_group_a,
                            non_orange_drivers,
                            assigned_non_orange,
                            leftover_non_orange,
                            label="Y/R"
                        )
                        new_count = len(indices_a)  # 대략적인 건수
                        print(f"[Y/R 2차 재배정] 그룹[{new_YR_group_a}] (약 {new_count}건) → [{driver_before['Type']}] 배정")
                        
                        group_centroids = recalc_group_centroids(df_shipping_region)
                    else:
                        print("[Y/R 2차 재배정] 배정 가능한 Y/R 버니가 더 이상 없습니다.")
                        break

                    print(f"클러스터 {label_a} 그룹 물량 40~55건이므로 [{new_YR_group_a}]에 배정")
                    group_stack.append(new_big_grp)
                    continue

                if 40 <= count_b <= 55 and count_a >= 60:
                    new_YR_group_b = f"{current_group}_Y/R_{label_b}"
                    new_big_grp = f"{current_group}_remain"

                    move_orders(df_shipping_region, indices_b, new_YR_group_b, clear_driver=False)
                    move_orders(df_shipping_region, indices_a, new_big_grp, clear_driver=False)

                    if assigned_non_orange < len(non_orange_drivers):
                        driver_before = non_orange_drivers.iloc[assigned_non_orange]
                        df_shipping_region, assigned_non_orange, leftover_non_orange = assign_driver_to_group(
                            df_shipping_region,
                            new_YR_group_b,
                            non_orange_drivers,
                            assigned_non_orange,
                            leftover_non_orange,
                            label="Y/R"
                        )
                        new_count = len(indices_b)
                        print(f"[Y/R 2차 재배정] 그룹[{new_YR_group_b}] (약 {new_count}건) → [{driver_before['Type']}] 배정")
                        
                        group_centroids = recalc_group_centroids(df_shipping_region)
                    else:
                        print("[Y/R 2차 재배정] 배정 가능한 Y/R 버니가 더 이상 없습니다.")
                        break

                    print(f"클러스터 {label_b} 그룹 물량 40~55건이므로 [{new_YR_group_b}]에 배정")
                    group_stack.append(new_big_grp)
                    continue

                if (40 <= count_a <= 55) and (40 <= count_b <= 55):
                    new_YR_group_a = f"{current_group}_Y/R_{label_a}"
                    new_YR_group_b = f"{current_group}_Y/R_{label_b}"

                    move_orders(df_shipping_region, indices_a, new_YR_group_a, clear_driver=False)
                    move_orders(df_shipping_region, indices_b, new_YR_group_b, clear_driver=False)

                    new_count_a = len(indices_a)
                    new_count_b = len(indices_b)

                    if assigned_non_orange < len(non_orange_drivers):
                        driver_before = non_orange_drivers.iloc[assigned_non_orange]
                        df_shipping_region, assigned_non_orange, leftover_non_orange = assign_driver_to_group(
                            df_shipping_region,
                            new_YR_group_a,
                            non_orange_drivers,
                            assigned_non_orange,
                            leftover_non_orange,
                            label="Y/R"
                        )
                        print(f"[Y/R 2차 재배정] 그룹[{new_YR_group_a}] (약 {new_count_a}건) {driver_before['Type']} 배정")
                        
                        group_centroids = recalc_group_centroids(df_shipping_region)
                    else:
                        print("[Y/R 2차 재배정] 배정 가능한 Y/R 버니가 더 이상 없습니다.")
                        break

                    if assigned_non_orange < len(non_orange_drivers):
                        driver_before = non_orange_drivers.iloc[assigned_non_orange]
                        df_shipping_region, assigned_non_orange, leftover_non_orange = assign_driver_to_group(
                            df_shipping_region,
                            new_YR_group_b,
                            non_orange_drivers,
                            assigned_non_orange,
                            leftover_non_orange,
                            label="Y/R"
                        )
                        print(f"[Y/R 2차 재배정] 그룹[{new_YR_group_b}] (약 {new_count_b}건) {driver_before['Type']} 배정")
                        
                        group_centroids = recalc_group_centroids(df_shipping_region)
                    else:
                        print("[Y/R 2차 재배정] 배정 가능한 Y/R 버니가 더 이상 없습니다.")
                        break

                    leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
                    print(f"[Y/R 2차 재배정] 남은 Y/R 버니: {leftover_non_orange}명")
                    continue

                if 40 <= count_a <= 55 and 20 <= count_b < 40:
                    new_YR_group_a = f"{current_group}_Y/R_{label_a}"
                    new_group_b = f"{current_group}_R_{label_b}"

                    move_orders(df_shipping_region, indices_a, new_YR_group_a, clear_driver=False)
                    move_orders(df_shipping_region, indices_b, new_group_b, clear_driver=False)

                    if assigned_non_orange < len(non_orange_drivers):
                        driver_before = non_orange_drivers.iloc[assigned_non_orange]
                        df_shipping_region, assigned_non_orange, leftover_non_orange = assign_driver_to_group(
                            df_shipping_region,
                            new_YR_group_a,
                            non_orange_drivers,
                            assigned_non_orange,
                            leftover_non_orange,
                            label="Y/R"
                        )
                        print(f"[Y/R 2차 재배정] 그룹[{new_YR_group_a}] (약 {len(indices_a)}건) {driver_before['Type']} 배정")
                        
                        group_centroids = recalc_group_centroids(df_shipping_region)
                    else:
                        print("[Y/R 2차 재배정] 배정 가능한 Y/R 버니가 더 이상 없습니다.")
                        break

                    leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
                    print(f"[Y/R 2차 재배정] 남은 Y/R 버니: {leftover_non_orange}명")
                    continue

                if 40 <= count_b <= 55 and 20 <= count_a < 40:
                    new_YR_group_b = f"{current_group}_Y/R_{label_b}"
                    new_group_a = f"{current_group}_R_{label_a}"

                    move_orders(df_shipping_region, indices_b, new_YR_group_b, clear_driver=False)
                    move_orders(df_shipping_region, indices_a, new_group_a, clear_driver=False)

                    if assigned_non_orange < len(non_orange_drivers):
                        driver_before = non_orange_drivers.iloc[assigned_non_orange]
                        df_shipping_region, assigned_non_orange, leftover_non_orange = assign_driver_to_group(
                            df_shipping_region,
                            new_YR_group_b,
                            non_orange_drivers,
                            assigned_non_orange,
                            leftover_non_orange,
                            label="Y/R"
                        )
                        print(f"[Y/R 2차 재배정] 그룹[{new_YR_group_b}] (약 {len(indices_b)}건) → [{driver_before['Type']}] 배정")

                        group_centroids = recalc_group_centroids(df_shipping_region)
                    else:
                        print("[Y/R 2차 재배정] 배정 가능한 Y/R 버니가 더 이상 없습니다.")
                        break

                    leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
                    print(f"[Y/R 2차 재배정] 남은 Y/R 버니: {leftover_non_orange}명")
                    continue

                if leftover_non_orange <= 0:
                    print("더 이상 Y/R 버니가 없습니다.")
                    stop_all = True
                    break

                current = cluster_counts[larger_cluster]
                if current < 40:
                    deficit = 40 - current
                    print(f"  → 부족: {deficit}건 필요")

                    # smaller 클러스터 주문들
                    smaller_orders = grp_orders[grp_orders['cluster'] == smaller_cluster].copy()

                    larger_points = grp_orders[grp_orders['cluster'] == larger_cluster][['lat','lng']]
                    smaller_orders['dist_to_larger'] = smaller_orders.apply(
                        lambda row: min_dist_to_group(row, larger_points),
                        axis=1
                    )
                    smaller_orders.sort_values('dist_to_larger', inplace=True)
                    indices_to_move = smaller_orders.index[:deficit]

                    print(f"  → {deficit}건을 smaller 클러스터에서 larger 클러스터로 이동")
                    # driver 컬럼 초기화 후 이동
                    move_orders(df_shipping_region, indices_to_move, current_group, clear_driver=True)
                    grp_orders.loc[indices_to_move, 'cluster'] = larger_cluster
                    
                elif current > 40:
                    surplus = current - 40
                    print(f"  → 초과: {surplus}건 제거 필요")

                    larger_orders = grp_orders[grp_orders['cluster'] == larger_cluster].copy()
                    smaller_points = grp_orders[grp_orders['cluster'] == smaller_cluster][['lat','lng']]

                    larger_orders['dist_to_small'] = larger_orders.apply(
                        lambda row: min_dist_to_group(row, smaller_points),
                        axis=1
                    )
                    larger_orders.sort_values('dist_to_small', inplace=True)
                    indices_to_move = larger_orders.index[:surplus]

                    print(f"  → {surplus}건을 큰 클러스터에서 작은 클러스터로 이동")
                    move_orders(df_shipping_region, indices_to_move, current_group, clear_driver=True)
                    grp_orders.loc[indices_to_move, 'cluster'] = smaller_cluster
                    
                # 다시 갱신된 클러스터 수 체크(실제로는 df_shipping_region 변경)
                new_counts_ = grp_orders['cluster'].value_counts()
                new_larger_count = new_counts_.get(larger_cluster, 0)
                print(f"  → 조정 후: 큰 클러스터 {larger_cluster} 주문수: {new_larger_count} (목표:40)")

                # 만약 정확히 40이 됐다면 → Y/R 배정
                if new_larger_count == 40:
                    base_name = current_group.split('_cluster')[0]
                    yr_group_counter.setdefault(base_name, 0)
                    yr_group_counter[base_name] += 1

                    new_grp_label_large = f"{base_name}_cluster_Y/R_{yr_group_counter[base_name]}"
                    new_grp_label_small = f"{current_group}_remaining"

                    indices_large = grp_orders[grp_orders['cluster'] == larger_cluster].index
                    indices_small = grp_orders[grp_orders['cluster'] == smaller_cluster].index

                    # 그룹 이동
                    move_orders(df_shipping_region, indices_large, new_grp_label_large, clear_driver=False)
                    move_orders(df_shipping_region, indices_small, new_grp_label_small, clear_driver=False)

                    if assigned_non_orange < len(non_orange_drivers):
                        driver_before = non_orange_drivers.iloc[assigned_non_orange]
                        df_shipping_region, assigned_non_orange, leftover_non_orange = assign_driver_to_group(
                            df_shipping_region,
                            new_grp_label_large,
                            non_orange_drivers,
                            assigned_non_orange,
                            leftover_non_orange,
                            label="Y/R"
                        )
                        print(f"[Y/R 2차 재배정] 그룹[{new_grp_label_large}] (약 {new_larger_count}건) → [{driver_before['Type']}] 배정")

                        group_centroids = recalc_group_centroids(df_shipping_region)
                    else:
                        print("[Y/R 2차 재배정] 배정 가능한 Y/R 버니가 더 이상 없습니다.")
                        break

                print("[Y/R 2차 재배정] 후 그룹별 물량:\n", 
                    df_shipping_region.groupby('group')['shipping_uuid'].count())

                if leftover_non_orange <= 0:
                    break

            if stop_all:
                break

        leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
        print(f"[Y/R 2차 재배정] 남은 Y/R 버니: {leftover_non_orange}명")

        if stop_all:
            break

    print("[Y/R 2차 재배정] 로직 완료 or 버니 부족")

    # ---------------------------------------------------------------------------
    # 3차 배정: Y/R 3차 재배정 (Assign Y/R drivers to groups with 40~55 orders left unassigned)
    print("### [Y/R 3차 배정] ###")
    for _ in range(len(non_orange_drivers)):
        if leftover_non_orange <= 0:
            print("[3차 배정] No leftover Y/R drivers. Terminating.")
            break
        unassigned_df = df_shipping_region[df_shipping_region['driver_type'].isna()]
        unassigned_group_counts = unassigned_df.groupby('group')['shipping_uuid'].count()
        y_r_groups = unassigned_group_counts[(unassigned_group_counts >= 40) & (unassigned_group_counts <= 55)].index.tolist()
        print(f"[3차 배정] Unassigned groups: {y_r_groups}")
        
        for grp in y_r_groups:
            if leftover_non_orange <= 0:
                print("[3차 배정] No leftover Y/R drivers in 40~55 groups. Terminating.")
                break
            new_grp_label = f"{grp}_Y/R"
            indices = df_shipping_region[df_shipping_region['group'] == grp].index
            df_shipping_region.loc[indices, 'group'] = new_grp_label
            driver = non_orange_drivers.iloc[assigned_non_orange]
            df_shipping_region.loc[df_shipping_region['group'] == new_grp_label, 'driver_type'] = driver['Type']
            print(f"[3차 배정] Group {new_grp_label} (orders: {unassigned_group_counts[grp]}) assigned to Y/R driver {driver['uuid']}.")
            assigned_non_orange += 1
            leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
            group_centroids = recalc_group_centroids(df_shipping_region)
            print("[3차 배정] Updated group counts:\n", df_shipping_region.groupby('group')['shipping_uuid'].count())
        leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
        print(f"[3차 배정] Leftover Y/R drivers: {leftover_non_orange}")

    # ---------------------------------------------------------------------------
    # 4차 배정: Y/R 4차 재배정
    print("### [Y/R 4차 재배정] ###")
    for _ in range(len(non_orange_drivers)):
        if leftover_non_orange <= 0:
            print("남은 버니 부족으로 종료")
            break
        unassigned_groups = df_shipping_region[df_shipping_region['driver_type'].isna()]['group'].unique()
        if len(unassigned_groups) == 0:
            print("[Y/R 4차 재배정] 배정되지 않은 그룹이 없습니다.")
            break
        unassigned_group_counts = (df_shipping_region[df_shipping_region['group'].isin(unassigned_groups)]
                                .groupby('group')['shipping_uuid'].count())
        
        unassigned_max_group = unassigned_group_counts.idxmax()
        print(f"선택된 unassigned_max_group (물량 많은 그룹): {unassigned_max_group} (물량: {unassigned_group_counts[unassigned_max_group]}건)")
        
        if unassigned_max_group in group_centroids:
            unassigned_max_group_centroid = group_centroids[unassigned_max_group]
        else:
            orders = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]
            unassigned_max_group_centroid = (orders['lat'].mean(), orders['lng'].mean())
            group_centroids[unassigned_max_group] = unassigned_max_group_centroid

        unassigned_group_orders = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]
        plus_candidate_groups = [g for g in group_centroids.keys() if g != unassigned_max_group]
        
        if plus_candidate_groups:
            distances = {g: np.sqrt((unassigned_max_group_centroid[0] - group_centroids[g][0])**2 +
                                (unassigned_max_group_centroid[1] - group_centroids[g][1])**2)
                        for g in plus_candidate_groups}
            nearest_group = min(distances, key=distances.get)
            print(f"unassigned_max_group 과 가장 가까운 그룹: {nearest_group}")
            
            nearest_orders = df_shipping_region[df_shipping_region['group'] == nearest_group]
            nearest_driver_type = nearest_orders['driver_type'].iloc[0] if not nearest_orders.empty else None
            print(f"근접 그룹 {nearest_group}의 driver_type: {nearest_driver_type}")
            
            if nearest_group not in group_centroids:
                orders_ng = df_shipping_region[df_shipping_region['group'] == nearest_group]
                group_centroids[nearest_group] = (orders_ng['lat'].mean(), orders_ng['lng'].mean())
            
            volume = unassigned_group_counts[unassigned_max_group]
            nearest_volume = nearest_orders['shipping_uuid'].count()
            
            if pd.isna(nearest_driver_type):
                print(f"근접 그룹 {nearest_group}의 driver_type이 null이므로, {unassigned_max_group}의 물량이 40이 될 때까지 데이터를 이동합니다.")
                if volume < 40:
                    temp = nearest_orders.copy()
                    unassigned_points = unassigned_group_orders[['lat','lng']]
                    # 여기서는 거리 계산을 위한 새로운 컬럼 "dist"를 생성
                    temp["dist"] = temp.apply(lambda r: min_dist_to_group(r, unassigned_points), axis=1)

                    temp.sort_values("dist", inplace=True)
                    needed = 40 - volume
                    available = nearest_volume - 20  # 이동 가능한 최대 주문 수
                    move_count = min(needed, available)
                    indices_to_move = temp.index[:move_count]  # 직접 인덱스 선택
                    if  needed > available:
                        print("이동할 데이터가 없으므로, [Y/R 4차 재배정]을 종료합니다.")
                        break

                    move_orders(df_shipping_region, indices_to_move, unassigned_max_group, clear_driver=True)
                    unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                    print(f"→ 이동 후 {unassigned_max_group}의 물량: {unassigned_max_group_volume}건")
                    if unassigned_max_group_volume >= 40:
                        if assigned_non_orange < len(non_orange_drivers):
                            driver = non_orange_drivers.iloc[assigned_non_orange]
                            df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_type'] = driver['Type']
                            new_name = f"{unassigned_max_group}_Y/R"
                            df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name
                            
                            df_shipping_region, assigned_non_orange, leftover_non_orange = assign_driver_to_group(
                            df_shipping_region,
                            new_name,
                            non_orange_drivers,
                            assigned_non_orange,
                            leftover_non_orange,
                            label="Y/R"
                            )
                            print(f"[Y/R 4차 재배정] 그룹({new_name})에 {driver['Type']} 배정.")
                            group_centroids = recalc_group_centroids(df_shipping_region)
                        else:
                            print("[Y/R 4차 재배정] 배정 가능한 Y/R 버니 없음.")
                            break
                    else:
                        print(f"이동할 수 있는 주문이 부족 (필요: {needed}, 이동 가능: {move_count}) → 종료")
                        break

                elif volume >= 50:
                    temp = unassigned_group_orders.copy()
                    nearest_group_points = df_shipping_region[df_shipping_region['group'] == nearest_group][['lat','lng']]
                    temp["dist"] = temp.apply(lambda row: min_dist_to_group(row, nearest_group_points), axis=1)
                    temp.sort_values("dist", inplace=True)
                    needed = volume - 49
                    indices_to_move = temp.index[:needed]
                    print(f"unassigned_max_group 에 이동할 주문 수: {len(indices_to_move)}")
                    if len(indices_to_move) == 0:
                        print("이동할 데이터가 없으므로 종료")
                        break

                    move_orders(df_shipping_region, indices_to_move, nearest_group, clear_driver=True)
                    unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                    if unassigned_max_group_volume <= 49:
                        if assigned_non_orange < len(non_orange_drivers):
                            driver = non_orange_drivers.iloc[assigned_non_orange]
                            df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_type'] = driver['Type']
                            new_name = f"{unassigned_max_group}_Y/R"
                            df_shipping_region, assigned_non_orange, leftover_non_orange = assign_driver_to_group(
                            df_shipping_region,
                            new_name,
                            non_orange_drivers,
                            assigned_non_orange,
                            leftover_non_orange,
                            label="Y/R"
                            )
                            print(f"[Y/R 4차 재배정] 그룹({new_name})에 {driver['Type']} 배정.")
                            group_centroids = recalc_group_centroids(df_shipping_region)
                        else:
                            print("[Y/R 4차 재배정] 배정 가능한 Y/R 버니 없음.")
                            break
                    else:
                        print("이동할 데이터가 부족 → 종료")
                        break

            elif nearest_driver_type in ['YELLOW', 'RAINBOW']:
                if nearest_volume < 40:
                    print(f"근접 그룹 {nearest_group}의 주문 수가 40건 미만({nearest_volume}건) → 종료")
                    break
                else:
                    print(f"근접 그룹 {nearest_group}의 driver_type이 Y/R입니다. {unassigned_max_group}의 물량이 40이 될 때까지 이동 시도합니다.")
                    
                    if volume < 40:
                        needed = 40 - volume
                        available = nearest_volume - 40
                        move_count = min(needed, available)
                        print(f"{unassigned_max_group}: 필요 {needed}건; {nearest_group}에서 최대 {move_count}건 이동 가능 (volume {nearest_volume}).")
                        if needed > available:
                            print("이동할 데이터 부족 → 종료")
                            break
                        temp = nearest_orders.copy()
                        unassigned_points = unassigned_group_orders[['lat','lng']]
                        temp["dist"] = temp.apply(lambda r: min_dist_to_group(r, unassigned_points), axis=1)

                        temp.sort_values("dist", inplace=True)
                        indices_to_move = temp.index[:move_count]
                        print(f"[Y/R 4차 재배정] 이동할 주문 수: {len(indices_to_move)}")
                        if needed > available:
                            print("이동할 데이터 부족 → 종료")
                            break

                        move_orders(df_shipping_region, indices_to_move, unassigned_max_group, clear_driver=True)
                        unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                        if unassigned_max_group_volume >= 40:
                            if assigned_non_orange < len(non_orange_drivers):
                                driver = non_orange_drivers.iloc[assigned_non_orange]
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_type'] = driver['Type']
                                new_name = f"{unassigned_max_group}_Y/R"
                                df_shipping_region, assigned_non_orange, leftover_non_orange = assign_driver_to_group(
                                df_shipping_region,
                                new_name,
                                non_orange_drivers,
                                assigned_non_orange,
                                leftover_non_orange,
                                label="Y/R"
                                )
                                print(f"[Y/R 4차 재배정] 그룹({new_name})에 {driver['Type']} 배정.")
                                group_centroids = recalc_group_centroids(df_shipping_region)
                            else:
                                print("[Y/R 4차 재배정] 배정 가능한 Y/R 버니 없음.")
                                break
                        else:
                            print(f"이동할 데이터 부족 (필요: {needed}, 이동 가능: {move_count}) → 종료")
                            break
                    # nearest_group에게 넘기기
                    elif volume >= 56:
                        if nearest_group in group_centroids:
                            nearest_group_centroid = group_centroids[nearest_group]
                        else:
                            nearest_group_orders = df_shipping_region[df_shipping_region['group'] == nearest_group]
                            nearest_group_centroid = (nearest_group_orders['lat'].mean(), nearest_group_orders['lng'].mean())
                            group_centroids[nearest_group] = nearest_group_centroid

                        temp = df_shipping_region[df_shipping_region['group'] == unassigned_max_group].copy()
                        
                        nearest_group_points = df_shipping_region[df_shipping_region['group'] == nearest_group][['lat','lng']]
                        temp['dist_to_A'] = temp.apply(lambda row: min_dist_to_group(row, nearest_group_points), axis=1)
                        
                        temp.sort_values('dist_to_A', inplace=True)
                        # 60이상은 위에서 이미 클러스터링 돼서 남아있는 그룹은 50~59
                        needed = volume - 55
                        available = 55 - nearest_volume  # 이동 가능한 최대 주문 수 => 배정이 안 될 수도 있으므로, 여유 둠.(최후)
                        move_count = min(needed, available)
                        orders_to_move = temp.head(move_count)
                        print(f"unassigned_max_group 에 이동할 주문 수: {len(orders_to_move)}")

                        if  needed > available:
                            print("이동할 데이터가 없으므로, [Y/R 4차 재배정]을 종료합니다.")
                            break

                        move_orders(df_shipping_region, orders_to_move.index, nearest_group, clear_driver=True)

                         # (3) 만약 nearest_group에 (Y/R 등)가 이미 배정되어 있다면, 그 버니정보를 이동된 주문에도 적용
                        driver_data = df_shipping_region.loc[
                            (df_shipping_region['group'] == nearest_group)
                            & (df_shipping_region['driver_type'].notna())
                        ].head(1)

                        if not driver_data.empty:
                            assigned_type = driver_data['driver_type'].iloc[0]
                            assigned_code = driver_data['driver_code'].iloc[0]

                            df_shipping_region.loc[orders_to_move.index, 'driver_type'] = assigned_type
                            df_shipping_region.loc[orders_to_move.index, 'driver_code'] = assigned_code

                        # 이후 물량 카운트 버니 배정 처리
                        unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                        nearest_volume = df_shipping_region[df_shipping_region['group'] == nearest_group]['shipping_uuid'].count()
                        print(f"이동 후, unassigned_group({unassigned_max_group})={unassigned_max_group_volume}건, nearest_group({nearest_group})={nearest_volume}건")


                        if unassigned_max_group_volume <= 55:
                            if assigned_non_orange < len(non_orange_drivers):
                                driver = non_orange_drivers.iloc[assigned_non_orange]
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_type'] = driver['Type']
                                new_name = f"{unassigned_max_group}_Y/R"
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name                                                      
                                
                                df_shipping_region, assigned_non_orange, leftover_non_orange = assign_driver_to_group(
                                df_shipping_region,
                                new_name,
                                non_orange_drivers,
                                assigned_non_orange,
                                leftover_non_orange,
                                label="Y/R"
                                )
                                print(f"[4차 Y/R 배정] 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, [{driver['Type']}]에 배정합니다.")
                                group_centroids = recalc_group_centroids(df_shipping_region)
                        else:
                            print(f"이동할 수 있는 물량이 부족해 종료, 필요물량: {needed}, 가져올 수 있는 물량:{move_count}")
                            break

            elif nearest_driver_type == 'ORANGE':
                if nearest_volume < 20:
                    print(f"근접 그룹 {nearest_group}의 주문 수가 20건 미만({nearest_volume}건) → 종료")
                    break
                else:
                    print(f"근접 그룹 {nearest_group}의 driver_type이 ORANGE입니다. {unassigned_max_group}의 물량이 20이 될 때까지 이동 시도합니다.")
                    if volume < 40:
                        temp = nearest_orders.copy()

                        unassigned_points = unassigned_group_orders[['lat','lng']]
                        temp['dist_to_A'] = temp.apply(lambda r: min_dist_to_group(r, unassigned_points), axis=1)
                        
                        temp.sort_values('dist_to_A', inplace=True)
                        needed = 40 - volume
                        available = nearest_volume - 20  # 이동 가능한 최대 주문 수
                        move_count = min(needed, available)
                        orders_to_move = temp.head(move_count)
                        print(f"unassigned_max_group 에 이동할 주문 수: {len(orders_to_move)}")

                        if  needed > available:
                            print("이동할 데이터가 없으므로, [Y/R 4차 재배정]을 종료합니다.")
                            break

                        move_orders(df_shipping_region, orders_to_move.index, unassigned_max_group, clear_driver=True)

                        unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                        if unassigned_max_group_volume >= 40:
                            if assigned_non_orange < len(non_orange_drivers):
                                driver = non_orange_drivers.iloc[assigned_non_orange]
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_type'] = driver['Type']
                                new_name = f"{unassigned_max_group}_Y/R"
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name
                                df_shipping_region, assigned_non_orange, leftover_non_orange = assign_driver_to_group(
                                df_shipping_region,
                                new_name,
                                non_orange_drivers,
                                assigned_non_orange,
                                leftover_non_orange,
                                label="Y/R"
                                )                                      
                                print(f"[4차 Y/R 배정] A 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, [{driver['Type']}]에 배정합니다.")
                                group_centroids = recalc_group_centroids(df_shipping_region)
                        else:
                            print(f"이동할 수 있는 물량이 부족해 종료, 필요물량: {needed}, 가져올 수 있는 물량:{move_count}")
                            break
                    
                    elif volume >= 56:
                        if nearest_group in group_centroids:
                            nearest_group_centroid = group_centroids[nearest_group]
                        else:
                            nearest_group_orders = df_shipping_region[df_shipping_region['group'] == nearest_group]
                            nearest_group_centroid = (nearest_group_orders['lat'].mean(), nearest_group_orders['lng'].mean())
                            group_centroids[nearest_group] = nearest_group_centroid

                        temp = df_shipping_region[df_shipping_region['group'] == unassigned_max_group].copy()

                        nearest_group_points = df_shipping_region[df_shipping_region['group'] == nearest_group][['lat','lng']]
                        temp['dist_to_A'] = temp.apply(lambda row: min_dist_to_group(row, nearest_group_points), axis=1)
                        
                        temp.sort_values('dist_to_A', inplace=True)
                        # 60이상은 위에서 이미 클러스터링 돼서 남아있는 그룹은 50~59
                        needed = volume - 55
                        available = 29 - nearest_volume  # 이동 가능한 최대 주문 수 => 배정이 안 될 수도 있으므로, 여유 둠.(최후)
                        move_count = min(needed, available)
                        orders_to_move = temp.head(move_count)
                        print(f"unassigned_max_group 에 이동할 주문 수: {len(orders_to_move)}")

                        if  needed > available:
                            print("이동할 데이터가 없으므로, [Y/R 4차 재배정]을 종료합니다.")
                            break

                        move_orders(df_shipping_region, orders_to_move.index, nearest_group, clear_driver=True)


                         # (3) 만약 nearest_group에
                         # 버니(Y/R 등)가 이미 배정되어 있다면, 그 버니정보를 이동된 주문에도 적용
                        driver_data = df_shipping_region.loc[
                            (df_shipping_region['group'] == nearest_group)
                            & (df_shipping_region['driver_type'].notna())
                        ].head(1)

                        if not driver_data.empty:
                            assigned_type = driver_data['driver_type'].iloc[0]
                            assigned_code = driver_data['driver_code'].iloc[0]

                            df_shipping_region.loc[orders_to_move.index, 'driver_type'] = assigned_type
                            df_shipping_region.loc[orders_to_move.index, 'driver_code'] = assigned_code

                        # 이후 물량 카운트, 버니 배정 처리
                        unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                        nearest_volume = df_shipping_region[df_shipping_region['group'] == nearest_group]['shipping_uuid'].count()
                        print(f"이동 후, unassigned_group({unassigned_max_group})={unassigned_max_group_volume}건, nearest_group({nearest_group})={nearest_volume}건")


                        if unassigned_max_group_volume <= 55:
                            if assigned_non_orange < len(non_orange_drivers):
                                driver = non_orange_drivers.iloc[assigned_non_orange]
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_type'] = driver['Type']
                                new_name = f"{unassigned_max_group}_Y/R"
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name                                                      
                                
                                df_shipping_region, assigned_non_orange, leftover_non_orange = assign_driver_to_group(
                                df_shipping_region,
                                new_name,
                                non_orange_drivers,
                                assigned_non_orange,
                                leftover_non_orange,
                                label="Y/R"
                                )
                                print(f"[4차 Y/R 배정] 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, [{driver['Type']}]에 배정합니다.")
                                group_centroids = recalc_group_centroids(df_shipping_region)
                        else:
                            print(f"이동할 수 있는 물량이 부족해 종료, 필요물량: {needed}, 가져올 수 있는 물량:{move_count}")
                            break
            
            else:
                print(f"근접 그룹 {nearest_group}의 driver_type({nearest_driver_type})에 대해 정의된 로직이 없습니다.")
        else:
            print("미배정 그룹 외에 다른 그룹이 없습니다.")

    # ---------------------------------------------------------------------------
    # 4차 배정: ORANGE 4차 재배정
    print("### [O 2차 재배정] ###")
    stop_all = False

    for _ in range(len(orange_drivers)):
        if leftover_orange <= 0:
            stop_all = True
            break

        unassigned_df = df_shipping_region[df_shipping_region['driver_type'].isna()]
        
        # 현재 group별 주문 수 집계
        group_counts = unassigned_df.groupby('group')['shipping_uuid'].count()
        # 40건 이상인 그룹만 추출
        candidate_groups = group_counts[group_counts >= 40].index.tolist()

        print(f"[O 2차 재배정] 남은 오렌지 버니: {leftover_orange}명, 배정되지 않은 40건 이상 그룹: {candidate_groups}")

        # 40건 이상 그룹이 없으면 종료
        if not candidate_groups:
            break

        # 40건 이상 그룹을 순회
        for grp in candidate_groups:
            if leftover_orange <= 0:
                stop_all = True
                break

            # 스택(LIFO)으로 관리
            group_stack = [grp]

            while group_stack and not stop_all:
                current_group = group_stack.pop()
                grp_orders = df_shipping_region[df_shipping_region['group'] == current_group].copy()
                current_count = grp_orders.shape[0]

                # 이미 40건 미만으로 줄었다면 스킵
                if current_count < 40:
                    continue

                print(f"[O 2차 재배정] 그룹[{current_group}] (주문수: {current_count}) → KMeans 2클러스터링 시도")

                grp_orders, cluster_counts, label_a, count_a, label_b, count_b = \
                    split_group_kmeans_2clusters(df_shipping_region, current_group)

                # 만약 제대로 2개 클러스터가 안 나오면(=1개뿐)
                if len(cluster_counts) < 2 or label_b is None:
                    # 클러스터가 하나뿐이면 다음으로 넘어감
                    continue

                print(f"  → 클러스터 A={label_a}({count_a}건), 클러스터 B={label_b}({count_b}건)")

                # idx_a / idx_b 미리 추출
                idx_a = grp_orders[grp_orders['cluster'] == label_a].index
                idx_b = grp_orders[grp_orders['cluster'] == label_b].index

                # --------------------------------------
                # [조건1] 두 클러스터 모두 40건 이상
                # --------------------------------------
                if count_a >= 40 and count_b >= 40:
                    new_group_a = f"{current_group}_SPLIT_{label_a}"
                    new_group_b = f"{current_group}_SPLIT_{label_b}"

                    # move_orders 사용 (clear_driver=False)
                    move_orders(df_shipping_region, idx_a, new_group_a, clear_driver=False)
                    move_orders(df_shipping_region, idx_b, new_group_b, clear_driver=False)

                    print(f"  → 두 클러스터 모두 40건 이상 → [{new_group_a}], [{new_group_b}] 스택에 재추가")
                    group_stack.append(new_group_a)
                    group_stack.append(new_group_b)
                    continue

                # --------------------------------------
                # [조건2] 한쪽은 20~30, 다른 쪽은 40 이상
                # --------------------------------------
                if 20 <= count_a < 30 and count_b >= 40:
                    new_orange_group_a = f"{current_group}_O_{label_a}"
                    new_group_b = f"{current_group}_SPLIT_{label_b}"

                    move_orders(df_shipping_region, idx_a, new_orange_group_a, clear_driver=False)
                    move_orders(df_shipping_region, idx_b, new_group_b, clear_driver=False)

                    # 오렌지 배정
                    if assigned_orange < len(orange_drivers):
                        # 헬퍼 함수로 배정
                        driver_before = orange_drivers.iloc[assigned_orange]  # 배정 전 기사 정보
                        df_shipping_region, assigned_orange, leftover_orange = assign_driver_to_group(
                            df_shipping_region,
                            new_orange_group_a,
                            orange_drivers,
                            assigned_orange,
                            leftover_orange,
                            label="ORANGE"
                        )
                        print(f"  → [조건2] 그룹[{new_orange_group_a}]({count_a}건) → [{driver_before['Type']}] 배정")
                    else:
                        print("[2차 O 재배정] 더 이상 배정할 오렌지 버니가 없습니다.")
                        stop_all = True
                        break

                    group_stack.append(new_group_b)
                    continue

                if 20 <= count_b < 30 and count_a >= 40:
                    new_orange_group_b = f"{current_group}_O_{label_b}"
                    new_group_a = f"{current_group}_SPLIT_{label_a}"

                    move_orders(df_shipping_region, idx_b, new_orange_group_b, clear_driver=False)
                    move_orders(df_shipping_region, idx_a, new_group_a, clear_driver=False)

                    if assigned_orange < len(orange_drivers):
                        driver_before = orange_drivers.iloc[assigned_orange]
                        df_shipping_region, assigned_orange, leftover_orange = assign_driver_to_group(
                            df_shipping_region,
                            new_orange_group_b,
                            orange_drivers,
                            assigned_orange,
                            leftover_orange,
                            label="ORANGE"
                        )
                        print(f"  → [조건2] 그룹[{new_orange_group_b}]({count_b}건) → 오렌지 버니[{driver_before['Type']}] 배정")
                    else:
                        print("[2차 O 재배정] 더 이상 배정할 오렌지 버니가 없습니다.")
                        stop_all = True
                        break

                    group_stack.append(new_group_a)
                    continue

                # --------------------------------------
                # [조건3] 두 클러스터 모두 20~30
                # --------------------------------------
                if (20 <= count_a < 30) and (20 <= count_b < 30):
                    new_orange_group_a = f"{current_group}_O_{label_a}"
                    new_orange_group_b = f"{current_group}_O_{label_b}"

                    move_orders(df_shipping_region, idx_a, new_orange_group_a, clear_driver=False)
                    move_orders(df_shipping_region, idx_b, new_orange_group_b, clear_driver=False)

                    # A 배정
                    if assigned_orange < len(orange_drivers):
                        driver_a_before = orange_drivers.iloc[assigned_orange]
                        df_shipping_region, assigned_orange, leftover_orange = assign_driver_to_group(
                            df_shipping_region,
                            new_orange_group_a,
                            orange_drivers,
                            assigned_orange,
                            leftover_orange,
                            label="ORANGE"
                        )
                        print(f"  → [조건3] 그룹[{new_orange_group_a}]({count_a}건) → 오렌지 버니[{driver_a_before['Type']}] 배정")
                    else:
                        print("[2차 O 재배정] 더 이상 오렌지 버니가 없습니다.")
                        stop_all = True
                        break

                    # B 배정
                    if assigned_orange < len(orange_drivers):
                        driver_b_before = orange_drivers.iloc[assigned_orange]
                        df_shipping_region, assigned_orange, leftover_orange = assign_driver_to_group(
                            df_shipping_region,
                            new_orange_group_b,
                            orange_drivers,
                            assigned_orange,
                            leftover_orange,
                            label="ORANGE"
                        )
                        print(f"  → [조건3] 그룹[{new_orange_group_b}]({count_b}건) → 오렌지 버니[{driver_b_before['Type']}] 배정")
                    else:
                        print("[2차 O 재배정] 더 이상 오렌지 버니가 없습니다.")
                        stop_all = True
                        break

                    continue

                # --------------------------------------
                # [조건4] 한 클러스터가 20~30, 다른 클러스터가 30~40
                # --------------------------------------
                # 기존 in_range() 함수를 재사용
                if in_range(count_a, 20, 30) and in_range(count_b, 30, 40):
                    new_orange_group_a = f"{current_group}_O_{label_a}"
                    remain_group_b = f"{current_group}_remain_{label_b}"

                    move_orders(df_shipping_region, idx_a, new_orange_group_a, clear_driver=False)
                    move_orders(df_shipping_region, idx_b, remain_group_b, clear_driver=False)

                    if assigned_orange < len(orange_drivers):
                        driver_before = orange_drivers.iloc[assigned_orange]
                        df_shipping_region, assigned_orange, leftover_orange = assign_driver_to_group(
                            df_shipping_region,
                            new_orange_group_a,
                            orange_drivers,
                            assigned_orange,
                            leftover_orange,
                            label="ORANGE"
                        )
                        print(f"  → [조건4] 그룹[{new_orange_group_a}]({count_a}건) → [{driver_before['Type']}] 배정, 나머지[{remain_group_b}]는 유지({count_b}건)")
                    else:
                        print("[2차 O 재배정] 더 이상 오렌지 버니가 없습니다.")
                        stop_all = True
                        break

                    continue

                if in_range(count_b, 20, 30) and in_range(count_a, 30, 40):
                    new_orange_group_b = f"{current_group}_O_{label_b}"
                    remain_group_a = f"{current_group}_remain_{label_a}"

                    move_orders(df_shipping_region, idx_b, new_orange_group_b, clear_driver=False)
                    move_orders(df_shipping_region, idx_a, remain_group_a, clear_driver=False)

                    if assigned_orange < len(orange_drivers):
                        driver_before = orange_drivers.iloc[assigned_orange]
                        df_shipping_region, assigned_orange, leftover_orange = assign_driver_to_group(
                            df_shipping_region,
                            new_orange_group_b,
                            orange_drivers,
                            assigned_orange,
                            leftover_orange,
                            label="ORANGE"
                        )
                        print(f"  → [조건4] 그룹[{new_orange_group_b}]({count_b}건) → [{driver_before['Type']}] 배정, 나머지[{remain_group_a}]는 유지({count_a}건)")
                    else:
                        print("[2차 O 재배정] 더 이상 오렌지 버니가 없습니다.")
                        stop_all = True
                        break

                    continue

                # --------------------------------------
                # [조건5] 그 외 - 부족 / 초과 로직으로 20건 맞추기
                # --------------------------------------
                # 작은/큰 클러스터 식별
                if cluster_counts[label_a] <= cluster_counts[label_b]:
                    small_label, large_label = label_a, label_b
                else:
                    small_label, large_label = label_b, label_a

                small_count = cluster_counts[small_label]
                large_count = cluster_counts[large_label]

                print(f"  → [조건5] 작은 클러스터={small_label}({small_count}건), 큰 클러스터={large_label}({large_count}건)")

                # A) 작은 클러스터 < 20 → 큰 쪽에서 가져와 20 맞추기
                if small_count < 20:
                    deficit = 20 - small_count
                    print(f"    → 작은 클러스터 부족 {deficit}건. 큰 클러스터 -> 작은 클러스터 이동")

                    small_cluster_points = grp_orders[grp_orders['cluster'] == small_label][['lat','lng']]
                    large_orders = grp_orders[grp_orders['cluster'] == large_label].copy()
                    large_orders['dist_to_small'] = large_orders.apply(
                        lambda row: min_dist_to_group(row, small_cluster_points),
                        axis=1
                    )
                    large_orders.sort_values('dist_to_small', inplace=True)
                    indices_to_move = large_orders.index[:deficit]

                    # 실제 데이터프레임에 반영
                    df_shipping_region.loc[indices_to_move, ['driver_type', 'driver_code']] = np.nan
                    grp_orders.loc[indices_to_move, 'cluster'] = small_label
                    print(f"  → {deficit}건을 대형 클러스터에서 작은 클러스터로 이동 (가장 가까운 순)")

                # B) 작은 클러스터 >= 30 → 일부를 큰 쪽으로 이동
                elif small_count >= 30:
                    surplus = small_count - 20
                    print(f"    → 작은 클러스터 초과 {surplus}건. 작은 클러스터 -> 큰 클러스터 이동")

                    large_cluster_points = grp_orders[grp_orders['cluster'] == large_label][['lat','lng']]
                    small_orders = grp_orders[grp_orders['cluster'] == small_label].copy()
                    small_orders['dist_to_large'] = small_orders.apply(
                        lambda row: min_dist_to_group(row, large_cluster_points),
                        axis=1
                    )
                    small_orders.sort_values('dist_to_large', inplace=True)
                    indices_to_move = small_orders.index[:surplus]

                    df_shipping_region.loc[indices_to_move, ['driver_type', 'driver_code']] = np.nan
                    grp_orders.loc[indices_to_move, 'cluster'] = large_label
                    print(f"  → {surplus}건을 작은 클러스터에서 대형 클러스터로 이동")

                new_counts = grp_orders['cluster'].value_counts()
                new_small_count = new_counts.get(small_label, 0)
                print(f"  → 조정 후 작은 클러스터 {small_label} 주문 수: {new_small_count} (목표:20)")

                if new_small_count == 20:
                    base_name = current_group.split('_cluster')[0]
                    yr_group_counter.setdefault(base_name, 0)
                    yr_group_counter[base_name] += 1

                    new_grp_label_target = f"{base_name}_cluster_O_{yr_group_counter[base_name]}"
                    new_grp_label_remaining = f"{current_group}_remaining"

                    indices_target = grp_orders[grp_orders['cluster'] == small_label].index
                    indices_remaining = grp_orders[grp_orders['cluster'] == large_label].index

                    # 그룹 이동
                    move_orders(df_shipping_region, indices_target, new_grp_label_target, clear_driver=False)
                    move_orders(df_shipping_region, indices_remaining, new_grp_label_remaining, clear_driver=False)

                    if assigned_orange < len(orange_drivers):
                        driver_before = orange_drivers.iloc[assigned_orange]
                        df_shipping_region, assigned_orange, leftover_orange = assign_driver_to_group(
                            df_shipping_region,
                            new_grp_label_target,
                            orange_drivers,
                            assigned_orange,
                            leftover_orange,
                            label="ORANGE"
                        )
                        print(f"[2차 O 재배정] 그룹[{new_grp_label_target}] (주문수: {new_small_count}) → [{driver_before['Type']}] 배정")
                        group_centroids = recalc_group_centroids(df_shipping_region)

                        print("[2차 O 재배정] 후 그룹별 물량:\n", df_shipping_region.groupby('group')['shipping_uuid'].count())
                    else:
                        print("[2차 O 재배정] 배정 가능한 오렌지 버니가 더 이상 없습니다.")
                        break

                if leftover_orange <= 0:
                    print("[2차 O 재배정] 더 이상 오렌지 버니가 없습니다.")
                    stop_all = True
                    break

        leftover_orange = len(orange_drivers) - assigned_orange
        print(f"[O 2차 재배정] 루프 종료, 남은 오렌지 버니: {leftover_orange}")

        if stop_all:
            break

    print("[O 2차 재배정] 로직 종료 또는 버니 부족")

    # [3차 O 재배정]
    while leftover_orange > 0:
        # 남은 20-29 지역 남은 오렌지 버니에게 배정
        plus_unassigned_groups = df_shipping_region[df_shipping_region['driver_type'].isna()]['group'].unique()
        plus_candidate_groups = []
        for grp in plus_unassigned_groups:
            cnt = df_shipping_region[df_shipping_region['group'] == grp]['shipping_uuid'].count()
            if 20 <= cnt <= 29:
                plus_candidate_groups.append(grp)

        if not plus_candidate_groups:
            print("[3차 O 재배정] 더 이상 주문 수 20이상 29 이하의 미배정 그룹이 없습니다.")
            break

        print(f"[3차 O 재배정]대상 그룹: {plus_candidate_groups}")

        for grp in plus_candidate_groups:
            if leftover_orange <= 0:
                break
            
            grp_orders = df_shipping_region[df_shipping_region['group'] == grp]
            print("[3차 O 재배정] 20~29 그룹 Orange 에게 배정")
            
            # (1) 그룹 이름 변경
            new_grp_label = f"{grp}_O"
            selected_idx = grp_orders.index
            
            move_orders(df_shipping_region, selected_idx, new_grp_label, clear_driver=False)

            # (2) 오렌지 기사 배정
            if assigned_orange < len(orange_drivers):
                # 배정 전 기사 정보를 따로 보관
                driver_before = orange_drivers.iloc[assigned_orange]

                df_shipping_region, assigned_orange, leftover_orange = assign_driver_to_group(
                    df_shipping_region,
                    new_grp_label,
                    orange_drivers,
                    assigned_orange,
                    leftover_orange,
                    label="ORANGE"
                )
                # 기존 출력 메시지
                print(f"[3차 O 재배정] 그룹[{new_grp_label}] (물량={len(grp_orders)}) → [{driver_before['Type']}] 배정")

                group_centroids = recalc_group_centroids(df_shipping_region)
                
                # 기존 로직: 필요시 group_centroids에서 기존 grp 키를 삭제
                if grp in group_centroids:
                    del group_centroids[grp]
            else:
                print("[3차 O 재배정] 더 이상 배정할 오렌지 버니가 없습니다.")
                break
    # [4차 O 재배정]
    print("[ORANGE 4차 재배정]")
    while leftover_orange > 0:
        unassigned_groups = df_shipping_region[df_shipping_region['driver_type'].isna()]['group'].unique()
        if len(unassigned_groups) == 0:
            print("[ORANGE 4차 재배정] 배정되지 않은 그룹이 없습니다.")
            break

        unassigned_group_counts = (
            df_shipping_region[df_shipping_region['group'].isin(unassigned_groups)]
            .groupby('group')['shipping_uuid']
            .count()
        )
        
        # 물량이 가장 많은 그룹
        unassigned_max_group = unassigned_group_counts.idxmax()
        max_cnt = unassigned_group_counts[unassigned_max_group]
        print(f"선택된 unassigned_max_group (물량 많은 그룹): {unassigned_max_group} (물량: {max_cnt}건)")

        unassigned_group_orders = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]
        unassigned_points = unassigned_group_orders[['lat','lng']]

        plus_candidate_groups = [g for g in group_centroids.keys() if g != unassigned_max_group]
        if plus_candidate_groups:
            # 1) unassigned_max_group에서 각 candidate 그룹까지의 거리를 계산
            distances = {}
            for g in plus_candidate_groups:
                centroid = group_centroids[g]
                # 중심점 끼리의 거리 (혹은 unassigned_points의 평균 vs centroid)
                dist = np.sqrt((unassigned_points['lat'].mean() - centroid[0])**2
                            + (unassigned_points['lng'].mean() - centroid[1])**2)
                distances[g] = dist

            nearest_group = min(distances, key=distances.get)
            print(f"unassigned_max_group 과 가장 가까운 그룹: {nearest_group}")

            nearest_orders = df_shipping_region[df_shipping_region['group'] == nearest_group]
            nearest_points = nearest_orders[['lat','lng']] if not nearest_orders.empty else None
            
            # 근접 그룹의 driver_type
            if not nearest_orders.empty:
                nearest_driver_type = nearest_orders['driver_type'].iloc[0]
            else:
                nearest_driver_type = None

            print(f"근접 그룹 {nearest_group}의 driver_type: {nearest_driver_type}")

            # 근접 그룹 centroid 갱신 (혹시 없으면)
            if nearest_group not in group_centroids:
                nearest_group_centroids = (nearest_orders['lat'].mean(), nearest_orders['lng'].mean())
                group_centroids[nearest_group] = nearest_group_centroids

            volume = max_cnt  # unassigned_max_group 물량
            nearest_volume = len(nearest_orders)  # 근접 그룹 물량

            # -------------------------------------------------
            # 근접 그룹의 드라이버 타입이 NULL (pd.isna) 일 때
            # -------------------------------------------------
            if pd.isna(nearest_driver_type):
                print(f"근접 그룹 {nearest_group}의 driver_type이 null이므로, unassigned_max_group 의 물량이 20이 될 때까지 데이터를 이동합니다.")

                # A) 배정되지 않은 그룹의 수량이 20 미만
                if volume < 20:
                    needed = 20 - volume
                    available = nearest_volume - 20  # 이동 가능한 최대 주문 수
                    move_count = min(needed, available)
                    print(f"unassigned_max_group={unassigned_max_group} 부족분={needed}, 근접그룹={nearest_group} (volume={nearest_volume})에서 가져올 수={move_count}")

                    if needed > available:
                        print("이동할 데이터가 없으므로, [ORANGE 4차 재배정]을 종료합니다.")
                        break

                    temp = nearest_orders.copy()
                    temp['dist_to_A'] = temp.apply(
                        lambda r: min_dist_to_group(r, unassigned_points),
                        axis=1
                    )
                    temp.sort_values('dist_to_A', inplace=True)

                    orders_to_move = temp.head(move_count)
                    
                    move_orders(df_shipping_region, orders_to_move.index, unassigned_max_group, clear_driver=True)

                    # 이동 후 unassigned_max_group 물량 재확인
                    unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                    print(f"→ 이동 후 unassigned_max_group={unassigned_max_group}의 물량={unassigned_max_group_volume}건")

                    # 물량 20 이상 → Orange 배정
                    if unassigned_max_group_volume >= 20:
                        if assigned_orange < len(orange_drivers):
                            # 그룹명 변경
                            driver_before = orange_drivers.iloc[assigned_orange]
                            new_name = f"{unassigned_max_group}_O"
                            
                            # 그룹명 교체
                            df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name
                            
                            # 드라이버 배정
                            df_shipping_region, assigned_orange, leftover_orange = assign_driver_to_group(
                                df_shipping_region,
                                new_name,
                                orange_drivers,
                                assigned_orange,
                                leftover_orange,
                                label="ORANGE"
                            )
                            print(f"[ORANGE 4차 재배정] 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, 남은 O 버니[{driver_before['uuid']}]에 배정합니다.")

                            group_centroids = recalc_group_centroids(df_shipping_region)
                    else:
                        print(f"이동할 수 있는 물량이 부족해 종료, 필요물량: {needed}, 가져올 수 있는 물량:{move_count}")
                        break

                # B) 배정되지 않은 그룹의 물량이 30 이상
                elif volume >= 30:
                    needed = volume - 29
                    
                    # 거리 계산
                    temp = unassigned_group_orders.copy()
                    temp['dist_to_A'] = temp.apply(
                        lambda r: min_dist_to_group(r, nearest_points),
                        axis=1
                    )
                    temp.sort_values('dist_to_A', inplace=True)

                    orders_to_move = temp.head(needed)
                    print(f"nearest_group 에 이동할 주문 수: {len(orders_to_move)}")

                    if len(orders_to_move) == 0:
                        print("이동할 데이터가 없으므로 종료")
                        break

                    move_orders(df_shipping_region, orders_to_move.index, nearest_group, clear_driver=True)

                    unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                    nearest_group_volume = df_shipping_region[df_shipping_region['group'] == nearest_group].shape[0]
                    print(f"→ 이동 후 unassigned_max_group={unassigned_max_group}={unassigned_max_group_volume}건, nearest_group={nearest_group}={nearest_group_volume}건")

                    # 29 이하로 줄었다면 → Orange 배정
                    if unassigned_max_group_volume <= 29:
                        if assigned_orange < len(orange_drivers):
                            driver_before = orange_drivers.iloc[assigned_orange]
                            new_name = f"{unassigned_max_group}_O"
                            
                            # 그룹명 교체
                            df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name
                            
                            # 드라이버 배정
                            df_shipping_region, assigned_orange, leftover_orange = assign_driver_to_group(
                                df_shipping_region,
                                new_name,
                                orange_drivers,
                                assigned_orange,
                                leftover_orange,
                                label="ORANGE"
                            )
                            print(f"[ORANGE 4차 재배정] 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, [{driver_before['Type']}]에 배정합니다.")
                            
                            group_centroids = recalc_group_centroids(df_shipping_region)

            # -------------------------------------------------
            # 근접 그룹 드라이버 타입이 YELLOW / RAINBOW 일 때
            # -------------------------------------------------
            elif nearest_driver_type in ['YELLOW', 'RAINBOW']:
                if nearest_volume < 40:
                    print(f"근접 그룹 {nearest_group}의 물량이 40 미만({nearest_volume}건)이라 unassigned_max_group 에 데이터를 가져올 수 없습니다. 종료합니다.")
                    break
                else:
                    print(f"근접 그룹 {nearest_group}의 driver_type이 Y/R입니다. unassigned_max_group 목표 물량: 20~29, 데이터 이동 시도합니다.")

                    # A) unassigned_max_group 물량 < 20
                    if volume < 20:
                        needed = 20 - volume
                        available = nearest_volume - 40
                        move_count = min(needed, available)
                        print(f"unassigned_max_group={unassigned_max_group} 부족분={needed}, 근접그룹={nearest_group} (volume={nearest_volume})에서 가져올 수={move_count}")

                        if needed > available:
                            print("이동할 데이터가 없으므로, [ORANGE 4차 재배정]을 종료합니다.")
                            break

                        temp = nearest_orders.copy()
                        temp['dist_to_A'] = temp.apply(lambda r: min_dist_to_group(r, unassigned_points), axis=1)
                        temp.sort_values('dist_to_A', inplace=True)

                        orders_to_move = temp.head(move_count)
                        print(f"unassigned_max_group 에 이동할 주문 수: {len(orders_to_move)}")

                        if needed > available:
                            print("이동할 데이터가 없으므로, [ORANGE 4차 재배정]을 종료합니다.")
                            break

                        move_orders(df_shipping_region, orders_to_move.index, unassigned_max_group, clear_driver=True)

                        unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                        if unassigned_max_group_volume >= 20:
                            if assigned_orange < len(orange_drivers):
                                driver_before = orange_drivers.iloc[assigned_orange]
                                new_name = f"{unassigned_max_group}_O"
                                
                                # 그룹명 교체
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name
                                
                                # 드라이버 배정
                                df_shipping_region, assigned_orange, leftover_orange = assign_driver_to_group(
                                    df_shipping_region,
                                    new_name,
                                    orange_drivers,
                                    assigned_orange,
                                    leftover_orange,
                                    label="ORANGE"
                                )
                                print(f"[ORANGE 4차 배정] 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, [{driver_before['Type']}]에 배정합니다.")
                                
                                group_centroids = recalc_group_centroids(df_shipping_region)
                        else:
                            print(f"이동할 수 있는 물량이 부족해 종료, 필요물량: {needed}, 가져올 수 있는 물량:{move_count}")
                            break

                    # B) unassigned_max_group 물량 >= 30
                    elif volume >= 30:
                        needed = volume - 29
                        available = 55 - nearest_volume
                        move_count = min(needed, available)

                        if needed > available:
                            print("이동할 데이터가 없으므로, [O 4차 재배정]을 종료합니다.")
                            break

                        temp = df_shipping_region[df_shipping_region['group'] == unassigned_max_group].copy()
                        temp['dist_to_A'] = temp.apply(lambda r: min_dist_to_group(r, nearest_points), axis=1)
                        temp.sort_values('dist_to_A', inplace=True)

                        orders_to_move = temp.head(move_count)
                        print(f"nearest_group 에 이동할 주문 수: {len(orders_to_move)}")

                        move_orders(df_shipping_region, orders_to_move.index, nearest_group, clear_driver=True)

                        # 만약 nearest_group에 이미 배정된 드라이버가 있다면 적용
                        driver_data = df_shipping_region.loc[
                            (df_shipping_region['group'] == nearest_group)
                            & (df_shipping_region['driver_type'].notna())
                        ].head(1)
                        if not driver_data.empty:
                            assigned_type = driver_data['driver_type'].iloc[0]
                            assigned_code = driver_data['driver_code'].iloc[0]
                            df_shipping_region.loc[orders_to_move.index, 'driver_type'] = assigned_type
                            df_shipping_region.loc[orders_to_move.index, 'driver_code'] = assigned_code

                        unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                        nearest_volume = df_shipping_region[df_shipping_region['group'] == nearest_group]['shipping_uuid'].count()
                        print(f"이동 후, unassigned_group({unassigned_max_group})={unassigned_max_group_volume}건, nearest_group({nearest_group})={nearest_volume}건")

                        if unassigned_max_group_volume <= 29:
                            if assigned_orange < len(orange_drivers):
                                driver_before = orange_drivers.iloc[assigned_orange]
                                new_name = f"{unassigned_max_group}_O"
                                
                                # 그룹명 교체
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name
                                
                                # 드라이버 배정
                                df_shipping_region, assigned_orange, leftover_orange = assign_driver_to_group(
                                    df_shipping_region,
                                    new_name,
                                    orange_drivers,
                                    assigned_orange,
                                    leftover_orange,
                                    label="ORANGE"
                                )
                                print(f"[4차 ORANGE 배정] 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, [{driver_before['Type']}]에 배정합니다.")
                                
                                group_centroids = recalc_group_centroids(df_shipping_region)
                        else:
                            print(f"이동할 수 있는 물량이 부족해 종료, 필요물량: {needed}, 가져올 수 있는 물량:{move_count}")

            # -------------------------------------------------
            # 근접 그룹 드라이버 타입이 ORANGE 일 때
            # -------------------------------------------------
            elif nearest_driver_type == 'ORANGE':
                if nearest_volume < 20:
                    print(f"근접 그룹 {nearest_group}의 물량이 20 미만({nearest_volume}건)이라 unassigned_max_group 에 데이터를 가져올 수 없습니다. 종료합니다.")
                    break
                else:
                    print(f"근접 그룹 {nearest_group}의 driver_type이 Orange입니다. unassigned_max_group 의 물량이 20이 될 때까지 데이터 이동 시도합니다.")

                    # A) unassigned_max_group < 20
                    if volume < 20:
                        needed = 20 - volume
                        available = nearest_volume - 20
                        move_count = min(needed, available)

                        if needed > available:
                            print("이동할 데이터가 없으므로, [O 4차 재배정] 종료.")
                            break

                        temp = nearest_orders.copy()
                        unassigned_points = df_shipping_region[df_shipping_region['group'] == unassigned_max_group][['lat','lng']]
                        temp['dist_to_A'] = temp.apply(lambda row: min_dist_to_group(row, unassigned_points), axis=1)
                        temp.sort_values('dist_to_A', inplace=True)
                        orders_to_move = temp.head(move_count)
                        print(f"unassigned_max_group 에 이동할 주문 수: {len(orders_to_move)}")

                        move_orders(df_shipping_region, orders_to_move.index, unassigned_max_group, clear_driver=True)

                        unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                        nearest_volume = df_shipping_region[df_shipping_region['group'] == nearest_group]['shipping_uuid'].count()
                        print(f"이동 후, unassigned_group({unassigned_max_group})={unassigned_max_group_volume}건, nearest_group({nearest_group})={nearest_volume}건")

                        if unassigned_max_group_volume >= 20:
                            if assigned_orange < len(orange_drivers):
                                driver_before = orange_drivers.iloc[assigned_orange]
                                new_name = f"{unassigned_max_group}_O"
                                
                                # 그룹명 교체
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name
                                
                                # 드라이버 배정
                                df_shipping_region, assigned_orange, leftover_orange = assign_driver_to_group(
                                    df_shipping_region,
                                    new_name,
                                    orange_drivers,
                                    assigned_orange,
                                    leftover_orange,
                                    label="ORANGE"
                                )
                                print(f"[4차 ORANGE 배정] A 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, [{driver_before['Type']}]에 배정합니다.")

                                group_centroids = recalc_group_centroids(df_shipping_region)
                        else:
                            print(f"이동할 수 있는 물량이 부족해 종료, 필요물량: {needed}, 가져올 수 있는 물량:{move_count}")

                    # B) unassigned_max_group >= 30
                    elif volume >= 30:
                        needed = volume - 29
                        available = 55 - nearest_volume
                        move_count = min(needed, available)

                        if needed > available:
                            print("이동할 데이터가 없으므로, [O 4차 재배정]을 종료합니다.")
                            break

                        temp = df_shipping_region[df_shipping_region['group'] == unassigned_max_group].copy()
                        nearest_points = df_shipping_region[df_shipping_region['group'] == nearest_group][['lat','lng']]
                        temp['dist_to_A'] = temp.apply(lambda r: min_dist_to_group(r, nearest_points), axis=1)
                        temp.sort_values('dist_to_A', inplace=True)

                        orders_to_move = temp.head(move_count)
                        print(f"nearest_group 에 이동할 주문 수: {len(orders_to_move)}")

                        move_orders(df_shipping_region, orders_to_move.index, nearest_group, clear_driver=True)

                        driver_data = df_shipping_region.loc[
                            (df_shipping_region['group'] == nearest_group) &
                            (df_shipping_region['driver_type'].notna())
                        ].head(1)
                        if not driver_data.empty:
                            assigned_type = driver_data['driver_type'].iloc[0]
                            assigned_code = driver_data['driver_code'].iloc[0]

                            df_shipping_region.loc[orders_to_move.index, 'driver_type'] = assigned_type
                            df_shipping_region.loc[orders_to_move.index, 'driver_code'] = assigned_code

                        unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                        nearest_volume = df_shipping_region[df_shipping_region['group'] == nearest_group]['shipping_uuid'].count()
                        print(f"이동 후, unassigned_group({unassigned_max_group})={unassigned_max_group_volume}건, nearest_group({nearest_group})={nearest_volume}건")

                        if unassigned_max_group_volume <= 29:
                            if assigned_orange < len(orange_drivers):
                                driver_before = orange_drivers.iloc[assigned_orange]
                                new_name = f"{unassigned_max_group}_O"
                                
                                # 그룹명 교체
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name
                                
                                # 드라이버 배정
                                df_shipping_region, assigned_orange, leftover_orange = assign_driver_to_group(
                                    df_shipping_region,
                                    new_name,
                                    orange_drivers,
                                    assigned_orange,
                                    leftover_orange,
                                    label="ORANGE"
                                )
                                print(f"[4차 ORANGE 배정] 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, [{driver_before['Type']}]에 배정합니다.")

                                group_centroids = recalc_group_centroids(df_shipping_region)
                        else:
                            print(f"이동할 수 있는 물량이 부족해 종료, 필요물량: {needed}, 가져올 수 있는 물량:{move_count}")

            else:
                print(f"근접 그룹 {nearest_group}의 driver_type({nearest_driver_type})에 대해 정의된 로직이 없습니다.")
        else:
            print("미배정 그룹 외에 다른 그룹이 없습니다.")

    used_drivers_idx = list(non_orange_drivers.index[:assigned_non_orange]) + list(orange_drivers.index[:assigned_orange])
    leftover_driver_df = fix_region_workflow_day_bunny_df.drop(used_drivers_idx)

    return df_shipping_region, leftover_driver_df

In [363]:

# ---------------------------------------------------------------------------
# 4) 지역별 처리
# ---------------------------------------------------------------------------
def process_region(
    region_name,
    zipcode_groups,
    df_shipping,
    workflow_day_bunny_df,
    leftover_all_fix_drivers,
):

    print(f"### {region_name} 처리 시작 ###")

    zipcode_groups = zipcode_groups[zipcode_groups['region']==region_name]

    # (2) 그룹화 및 데이터 저장
    result_gdf, df_shipping_region = group_and_map_zipcodes(zipcode_groups, df_shipping)

    # 지역별 bunny 기사 필터링
    region_workflow_day_bunny_df = workflow_day_bunny_df[workflow_day_bunny_df['Area'] == df_shipping_region.iloc[0, 3]].copy()

    # (3) 고정 기사(YELLOW/RAINBOW/ORANGE) & 화이트(WHITE) 기사 분리
    fix_region_workflow_day_bunny_df = region_workflow_day_bunny_df[
        region_workflow_day_bunny_df['Type'].isin(['YELLOW','RAINBOW','ORANGE'])
    ].copy()

    # 버니에게 배정 전 그룹별 물량
    df_shipping_region_group_count=df_shipping_region.groupby('group')['shipping_uuid'].agg('count').reset_index()
    print(f"{region_name} 초기 그룹별 물량\n{df_shipping_region_group_count}")

    fix_drivers = len(fix_region_workflow_day_bunny_df)

    if fix_drivers > 0:
        # 고정기사 배정
        df_shipping_region, leftover_driver_df = assign_fixed_drivers(df_shipping_region, fix_region_workflow_day_bunny_df)

        # leftover_all_fix_drivers 누적
        if not leftover_driver_df.empty:
            leftover_driver_df = leftover_driver_df.copy()
            leftover_driver_df['Region'] = region_name
            leftover_all_fix_drivers = pd.concat([leftover_all_fix_drivers, leftover_driver_df], ignore_index=True)

    # 미리 driver 컬럼이 없으면 생성
    for col in ['driver_type', 'driver_code']:
        if col not in df_shipping_region.columns:
            df_shipping_region[col] = np.nan

    unassigned_white_groups = df_shipping_region.groupby('group') \
        .filter(lambda x: x['driver_type'].isna().all()) \
        ['group'].unique()

    print("화이트 처리 대상 그룹:", unassigned_white_groups)
    for grp in unassigned_white_groups:
        df_shipping_region = isolate_white_clusters(df_shipping_region, grp)
        # 최종 결과 출력: 화이트 처리 후 그룹별 물량
        print("\n[화이트 처리 후 그룹별 물량]\n", df_shipping_region.groupby('group')['shipping_uuid'].count())
        
    print(f"{region_name} 처리 완료\n")
    return df_shipping_region, result_gdf, leftover_all_fix_drivers


In [364]:

# ---------------------------------------------------------------------------
# 5) 최종 실행 함수
# ---------------------------------------------------------------------------
def run_clustering(
    zipcode_groups,
    workflow_day_shipping_items_df,
    workflow_day_bunny_df,
):
    """
    전체 지역(zipcode_groups) 순회하며
    1) process_region() 호출 -> 기사 배정
    2) 최종적으로 지역별 DataFrame 통합
    3) leftover 기사를 별도 DataFrame에 저장
    4) 시각화 결과 HTML 생성 (visualize_clusters)
    """

    # 통합할 DF
    all_filtered_geo_dfs = pd.DataFrame()
    all_KMeans_dfs = pd.DataFrame()

    # 고정기사 DF
    leftover_all_fix_drivers = pd.DataFrame()


    # 원본 df_shipping 복사
    df_shipping = workflow_day_shipping_items_df.copy()

    # 지역별 처리
    for region_name in zipcode_groups['region'].unique():

        norm_region = normalize_region(region_name)
        region_df = df_shipping[df_shipping["Area"] == norm_region]
        
        if region_df.empty:
            print(
                f"스킵: {region_name} (정규화: {norm_region}) - 해당 지역 데이터 없음"
            )
            continue
        (
            df_shipping_region, 
            filtered_geo_df, 
            leftover_all_fix_drivers,
        ) = process_region(
            region_name=region_name,
            zipcode_groups=zipcode_groups,
            df_shipping=df_shipping,
            workflow_day_bunny_df=workflow_day_bunny_df,
            leftover_all_fix_drivers=leftover_all_fix_drivers,
        )
        
        # 처리된 주문은 원본에서 제거 (이미 배정된 주문)
        df_shipping = df_shipping[~df_shipping['shipping_uuid'].isin(df_shipping_region['shipping_uuid'])]

        # 지역별 결과 통합
        all_filtered_geo_dfs = pd.concat([all_filtered_geo_dfs, filtered_geo_df], ignore_index=True)
        all_KMeans_dfs = pd.concat([all_KMeans_dfs, df_shipping_region], ignore_index=True)

    # 남은 미배정 주문(df_shipping)도 합침
    all_KMeans_dfs = pd.concat([all_KMeans_dfs, df_shipping], ignore_index=True)


    # 그룹이 있는데 기사정보가 없는 행은 driver_type='WHITE'로 지정
    mask = (~all_KMeans_dfs['group'].isna()) & (all_KMeans_dfs['driver_type'].isna())
    all_KMeans_dfs.loc[mask, 'driver_type'] = 'WHITE'

    # 그룹이 없고, 기사정보가 없는 행은 driver_type = 'BLUE'로 지정 => 벌크
    mask1 = (all_KMeans_dfs['group'].isna()) & (all_KMeans_dfs['driver_type'].isna())
    all_KMeans_dfs.loc[mask1, 'driver_type'] = 'BLUE'

    # 1. driver_type 매핑 문자 정의
    type_mapping = {
        'WHITE': 'W',
        'RAINBOW': 'R',
        'YELLOW': 'Y',
        'ORANGE': 'O'
        # 나머지는 'B'로 처리
    }

    # 2. (Area, driver_type, group)가 모두 있는 행만 추출 (NaN은 제외)
    unique_rows = (
        all_KMeans_dfs[['Area', 'driver_type', 'group']]
        .dropna(subset=['Area','driver_type','group'])
        .drop_duplicates()
    )

    # 3. Area별 + driver_type별 순번을 관리할 dict
    dict_area_type_seq = {}  

    # 4. 최종 라벨 매핑 딕셔너리: dict_of_labels[(Area, driver_type, group)] = "강남W1" 등
    dict_of_labels = {}

    # unique_rows 순회하면서 순번 부여
    for _, row in unique_rows.iterrows():
        area_val = row['Area']
        d_type = row['driver_type']
        grp = row['group']

        # (Area, driver_type) 별로 seq 1부터 시작
        key_area_type = (area_val, d_type)
        if key_area_type not in dict_area_type_seq:
            dict_area_type_seq[key_area_type] = 1
        else:
            dict_area_type_seq[key_area_type] += 1

        seq = dict_area_type_seq[key_area_type]

        # driver_type이 매핑 사전에 없으면 'B'
        letter = type_mapping.get(d_type, 'B1')
        # 실제 라벨 형식: "강남-W1"
        label = f"{area_val}{letter}{seq}"

        # 해당 그룹(Area, driver_type, group)에 라벨 저장
        dict_of_labels[(area_val, d_type, grp)] = label

    # 각 행에서 (Area, driver_type, group)으로 dict_of_labels 조회
    for i in all_KMeans_dfs.index:
        area_val = all_KMeans_dfs.at[i, 'Area']
        d_type   = all_KMeans_dfs.at[i, 'driver_type']
        grp      = all_KMeans_dfs.at[i, 'group']

        key = (area_val, d_type, grp)
        # 매핑 딕셔너리에 있으면 그 라벨을, 없으면 기존 값을 유지
        if key in dict_of_labels:
            all_KMeans_dfs.at[i, 'cluster_label'] = dict_of_labels[key]
        else:
            if pd.notna(area_val) and pd.notna(d_type):
                letter = type_mapping.get(d_type, 'B1')
                all_KMeans_dfs.at[i, 'cluster_label'] = f"{area_val}{letter}"

    # leftover 기사 출력
    print("\n### 전체 지역에서 '배정받지 못한 고정 기사' 모음 ###")
    if leftover_all_fix_drivers.empty:
        print("모든 고정 기사 배정 완료(남은 기사 없음)")
    else:
        print(leftover_all_fix_drivers)


    return all_KMeans_dfs, leftover_all_fix_drivers


In [365]:
# ---------------------------------------------------------------------------
# 6) 실제 실행 
# ---------------------------------------------------------------------------
if __name__ == "__main__":


    workflow_day_shipping_items_df = workflow_day_shipping_items_df.copy()
    workflow_day_bunny_df = workflow_day_bunny_df.copy()
    
    workflow_day_shipping_items_df = workflow_day_shipping_items_df.astype({'zipcode': str})
    workflow_day_shipping_items_df['zipcode'] = workflow_day_shipping_items_df['zipcode'].astype(str).str.zfill(5) 
    

    date = workflow_day_bunny_df['Date'].drop_duplicates()
    year, month, day = map(int, date[0].split('-'))

    # BLUE 기사들의 Area 목록 추출
    blue_areas = list(workflow_day_bunny_df.loc[workflow_day_bunny_df['Type'] == 'BLUE', 'Area'])
   

    if is_weekend(year, month, day):
        df_regular['region'] = df_weekend['region'].apply(normalize_region)
        chosen_df_weekend = df_weekend[~df_weekend['region'].isin(blue_areas)]

        chosen_zipcode_groups = chosen_df_weekend
        chosen_zipcode_groups.reset_index(drop=True, inplace=True)
        
    else:
        df_regular['region'] = df_regular['region'].apply(normalize_region)
        chosen_df_regular = df_regular[~df_regular['region'].isin(blue_areas)]

        chosen_zipcode_groups = chosen_df_regular
        chosen_zipcode_groups.reset_index(drop=True, inplace=True)
    

    all_KMeans_dfs, leftover_drivers = run_clustering(
        zipcode_groups=chosen_zipcode_groups,
        workflow_day_shipping_items_df=workflow_day_shipping_items_df,
        workflow_day_bunny_df=workflow_day_bunny_df,
    )

    # 기사가 부족해 배정되지 않은 지역
    all_KMeans_dfs_left = all_KMeans_dfs[(~all_KMeans_dfs['group'].isna()) & (all_KMeans_dfs['driver_type'].isna())].reset_index(drop=True)

    # 전체 그룹들 중 20개 미만인 지역
    all_KMeans_dfs_20_less = all_KMeans_dfs.groupby('cluster_label').agg({'shipping_uuid': 'count'}).reset_index()
    all_KMeans_dfs_20_less_group = all_KMeans_dfs_20_less[all_KMeans_dfs_20_less['shipping_uuid']<20]

    # 드라이버 할당 되지 않은 그룹들중 20개 미만인 지역
    all_KMeans_dfs_sam = all_KMeans_dfs.groupby('group').agg({'shipping_uuid':'count', 'driver_type' : 'count'}).reset_index()
    all_KMeans_dfs_sam_20less=all_KMeans_dfs_sam[all_KMeans_dfs_sam['driver_type']==0]
    KMeans_dfs_none_driver_20less_group = all_KMeans_dfs_sam_20less[all_KMeans_dfs_sam_20less['shipping_uuid']<20]
    

    # 결과 확인
    print("\n=== 벌크 지역 ===")
    print(blue_areas)
    print("\n=== 배정받지 못한 고정 기사 ===")
    print(leftover_drivers)
    print("\n=== 배정되지 않은 그룹들 중 물량 20개 미만인 지역 ===")
    print(KMeans_dfs_none_driver_20less_group)
    print("\n=== 전체 그룹들 중 물량 20개 미만인 지역 ===")
    print(all_KMeans_dfs_20_less_group)

### 강북구 처리 시작 ###
강북구 초기 그룹별 물량
  group  shipping_uuid
0     A            105
화이트 처리 대상 그룹: ['A']
A의 주문 수가 105건

[화이트 처리] 그룹 A 총 주문 수: 105건. 클러스터링 시도...
클러스터링 결과: {0: 56, 1: 49}
[move_orders] Moved 56 orders to group [A_SPLIT_0].
[move_orders] Moved 49 orders to group [A_SPLIT_1].
→ 두 클러스터가 모두 40건 이상. 새로운 그룹 A_SPLIT_0, A_SPLIT_1 스택에 추가

[화이트 처리] 그룹 A_SPLIT_1 총 주문 수: 49건. 클러스터링 시도...
클러스터링 결과: {0: 28, 1: 21}
[move_orders] Moved 28 orders to group [A_SPLIT_1_WHITE_0].
[move_orders] Moved 21 orders to group [A_SPLIT_1_WHITE_1].
클러스터 초기 그룹 물량 20건이상 40건 미만이므로 그룹 A_SPLIT_1_WHITE_0, A_SPLIT_1_WHITE_1 driver_type='WHITE'로 업데이트

[화이트 처리] 그룹 A_SPLIT_0 총 주문 수: 56건. 클러스터링 시도...
클러스터링 결과: {0: 35, 1: 21}
[move_orders] Moved 35 orders to group [A_SPLIT_0_WHITE_0].
[move_orders] Moved 21 orders to group [A_SPLIT_0_WHITE_1].
클러스터 초기 그룹 물량 20건이상 40건 미만이므로 그룹 A_SPLIT_0_WHITE_0, A_SPLIT_0_WHITE_1 driver_type='WHITE'로 업데이트

[화이트 처리 후 그룹별 물량]
 group
A_SPLIT_0_WHITE_0    35
A_SPLIT_0_WHITE_1    21
A_SPLIT_1_

In [366]:
all_KMeans_dfs=all_KMeans_dfs.drop(columns='group')

In [367]:
# 각 그룹에 물량 집어넣기
# 각 실제 데이터(point)에 대해서 (lat, lng) 거리를 구하고, 가장 가까운 한 개 row의 group을 할당(가장 가까운 데이터의 group에 포함)

def overlap(overlap_df, all_KMeans_dfs):
    
    overlap_df = overlap_df.astype({'zipcode': str})
    overlap_df['zipcode'] = overlap_df['zipcode'].astype(str).str.zfill(5) 

    for col in ['driver_type', 'driver_code', 'cluster_label']:
        if col not in overlap_df.columns:
            overlap_df[col] = np.nan
            
    # 오버랩 데이터를 가장 가까이 있는 데이터의 그룹에 포함시키기
    for i, order_row in overlap_df.iterrows():
        area = order_row["Area"]
        lat_val = order_row["lat"]
        lng_val = order_row["lng"]
        
        
        # 1) Area가 같은 행만 추출
        area_slice = all_KMeans_dfs[all_KMeans_dfs["Area"] == area]
        
        # 2) 해당 지역(Area)에 데이터 자체가 전혀 없는 경우: 미배정 처리
        if area_slice.empty:
            overlap_df.at[i, "driver_type"]  = np.nan
            overlap_df.at[i, "driver_code"]  = np.nan
            overlap_df.at[i, "cluster_label"]  = np.nan
            
            print(f"[INFO] {area} 지역의 주문({order_row['shipping_uuid']})은 같은 지역 데이터가 없어 미배정 처리.")
            continue
        
        # 3) (lat, lng) 거리계산 -> 가장 가까운 group 찾기
        #    여기서는 유클리드로 비교
        area_slice["dist"] = np.sqrt((area_slice["lat"].astype(float) - lat_val)**2 + 
                                     (area_slice["lng"].astype(float) - lng_val)**2)
        
        # 가장 작은 dist를 갖는 row의 인덱스
        min_idx = area_slice["dist"].idxmin()
        nearest_row = area_slice.loc[min_idx]
        
        best_group_driver_type = nearest_row["driver_type"]
        
        # 그룹이 없을 경우 클러스터라벨 및 driver_type 추가하기
        cluster_label = all_KMeans_dfs[all_KMeans_dfs['Area'] == area][['cluster_label']].drop_duplicates()
        cluster_label = cluster_label.iloc[0]

        driver_type = all_KMeans_dfs[all_KMeans_dfs['Area'] == area][['driver_type']].drop_duplicates()
        driver_type = driver_type.iloc[0]
        

        # 혹시 nearest_row 자체에 driver_type이 BLUE로 들어가 있으면 BLUE 처리
        if best_group_driver_type == 'BLUE':
            overlap_df.at[i, "driver_type"]  = driver_type['driver_type']
            overlap_df.at[i, "driver_code"]  = np.nan
            overlap_df.at[i, "cluster_label"]  = cluster_label['cluster_label']
            
            print(f"[INFO] {area} 지역 주문({order_row['shipping_uuid']}) → 가장 가까운 데이터의 driver_type이 BLUE, BLUE 로 편입.")
            continue
        
        # 4) cluster_label 할당
        overlap_df.at[i, "cluster_label"] = cluster_label['cluster_label']
        
        best_group_cluster_label = nearest_row["cluster_label"]

        # 5) 배정된 그룹의 기사 정보 확인
        group_slice = all_KMeans_dfs[
            (all_KMeans_dfs["Area"] == area) & 
            (all_KMeans_dfs["cluster_label"] == best_group_cluster_label)
        ]

        unique_drivers = group_slice[["driver_type", "driver_code", "cluster_label"]].drop_duplicates()
        
        if len(unique_drivers) == 1:
            # 기사정보가 유일하게 한 명이면 그대로 배정
            row_driver = unique_drivers.iloc[0]
            overlap_df.at[i, "driver_type"]  = row_driver["driver_type"]
            overlap_df.at[i, "driver_code"]  = row_driver["driver_code"]
            overlap_df.at[i, "cluster_label"]  = row_driver["cluster_label"]
        else:
            # 기사가 여러 명이거나 아예 없다면 -> 일단 미배정
            overlap_df.at[i, "driver_type"]  = np.nan
            overlap_df.at[i, "driver_code"]  = np.nan
            overlap_df.at[i, "cluster_label"]  = np.nan
        
        print(f"[INFO] {area} 지역 주문({order_row['shipping_uuid']}) → 가장 가까운 데이터의 cluster_label=[{best_group_cluster_label}] 배정 완료")
    
    # 기존 all_KMeans_dfs + overlap_df를 합쳐 최종 반환
    all_KMeans_dfs_plus_overlap = pd.concat([all_KMeans_dfs, overlap_df], ignore_index=True)

    return all_KMeans_dfs_plus_overlap

In [368]:
# 오버랩 클러스터

overlap_df = pd.read_csv('../git_csv/20250329_오버랩데이터.csv')

all_KMeans_dfs_plus_overlap = overlap(overlap_df, all_KMeans_dfs)
today_date = datetime.now().strftime("%Y%m%d")
# 오버랩 최종 시각화
final_map = visualize_clusters(all_KMeans_dfs_plus_overlap)
final_map_filename = f"../우편번호그룹_시각화_html/{today_date}_오버랩포함_우편번호_그룹화_시각화.html"
test_final_map_filename = f"../우편번호그룹_시각화_html/20250329_토요일_오버랩포함_우편번호_그룹_시각화.html"
final_map.save(test_final_map_filename)
print(f"최종 통합 시각화 저장 완료: {test_final_map_filename}")


[INFO] 서대문 지역 주문(f9a29cf0520e40e7b0b02e4212ecd17a) → 가장 가까운 데이터의 cluster_label=[서대문W4] 배정 완료
[INFO] 양천 지역 주문(d2561f101975450da371f3673957d1bc) → 가장 가까운 데이터의 cluster_label=[양천W6] 배정 완료
[INFO] 분당 지역 주문(91c79533da8e4308a6fa61a53ddc9ce6) → 가장 가까운 데이터의 cluster_label=[분당W1] 배정 완료
[INFO] 계양 지역 주문(11d9ae5cd987462b9f1f6ba6acc7c591) → 가장 가까운 데이터의 cluster_label=[계양Y1] 배정 완료
[INFO] 서초가 지역 주문(a57b7115c9f84b85b107de57597cd505) → 가장 가까운 데이터의 cluster_label=[서초가W2] 배정 완료
[INFO] 부평 지역 주문(1566e5b9162f449b812445447330c6d9) → 가장 가까운 데이터의 cluster_label=[부평W3] 배정 완료
최종 통합 시각화 저장 완료: ../우편번호그룹_시각화_html/20250329_토요일_오버랩포함_우편번호_그룹_시각화.html
